# KNEIL virtual implant — stages 0 and 1
**Goal:** compute the signal the implant would record from real people's shank IMUs (the implant and the shank sensor
are on the same bone), so that existing datasets become implant-site data.

* **Stage 0** checks every dataset folder: found, format as expected, sensors, sampling rate, units.
* **Stage 1a** validates the method on Camargo: the virtual implant computed from each person's real shank IMU is compared
  with the implant signal from the replay of their measured joint angles (same point, same axes).
* **Stage 1b** checks consistency: two sensors on the same tibia must give the same implant signal
  (KneE-PAD: tibialis anterior vs gastrocnemius; WearGait-PD: lateral shank vs ankle).

Choose **Runtime → Run all**. Time: stage 0 about 2 min, stage 1a about 1–2 min per Camargo subject (unzipping from
Drive), stage 1b about 5 min. Every stage saves to `KNEIL_results/virtual_implant/`; stage 1a skips subjects already
done, so after a disconnect just run all again. **Send me** `stage01_log.txt` and `stage1_summary.png` from that folder.

In [ ]:
#@title Settings and Google Drive
DATA_ROOT = "/content/drive/MyDrive/Walking_Gesture_Datasets"  #@param {type:"string"}
CAMARGO_SUBJECTS = "all"  #@param {type:"string"}
TRIALS_PER_MODE = 4  #@param {type:"integer"}
KNEEPAD_SUBJECTS = 10  #@param {type:"integer"}
WEARGAIT_PEOPLE = 10  #@param {type:"integer"}
import os, sys
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
OUT = os.path.join(DATA_ROOT, 'KNEIL_results', 'virtual_implant')
os.makedirs(OUT, exist_ok=True)
LOG = os.path.join(OUT, 'stage01_log.txt')
SUBJECTS = 'all' if CAMARGO_SUBJECTS.strip().lower() == 'all' else [s.strip().lower() for s in CAMARGO_SUBJECTS.replace(',', ' ').split()]
print('data:', DATA_ROOT, '| results:', OUT)

In [ ]:
#@title Install packages (about 30 s)
!pip -q install mujoco mat-io
import mujoco, matio
print('mujoco', mujoco.__version__, '| mat-io ok')

In [ ]:
#@title Write `kneil_v1.py` — KNEIL body and implant model
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/kneil_v1.py', 'w').write(r'''"""
KNEIL motion simulation, v1 (walking).

Synthetic 3-axis acceleration at the tibial insert (device frame) for virtual subjects and
patient-like gait conditions, generated by kinematic replay in MuJoCo.

Chain:  virtual subject -> gait parameters -> one-stride pattern (published template, warped and
        modified) -> pelvis motion from stance-foot constraints -> trial with stride-to-stride
        variability -> MuJoCo replay at 1 kHz (specific force at the implant) -> heel-strike
        impact transient -> sensor model -> 100 Hz / +-8 g output with labels and ground truth.

Device frame (from the insert drawings): x anterior, z superior (normal to the insert),
y = z cross x = subject's left (medial in a right knee, lateral in a left knee).

Sources (details in SOURCES below and in the parameter functions):
  template motion   OpenSim example "subject01_walk1" (Gait2354 pipeline, opensim-models, GitHub)
  speed by age/sex  Bohannon & Andrews 2011 (Physiotherapy 97:182), comfortable gait speed
  stride vs speed   Alexander 1984 dimensionless relation, stride/leg = 2.3 * Froude^0.3
  post-TKA gait     Debbi et al. ISB 2013 (2 months after TKA: swing peak 41 deg, extension
                    deficit 8 deg, 0.72 m/s); Cushner et al. 2024 J Arthroplasty (implanted
                    tibial IMU, first 6 weeks: cadence +22 %, speed +50 %, stride +17 %)
  locked knee       hip hiking as main compensation, modest abduction (restricted-knee studies),
                    vaulting; amplitudes solved here so that the swing foot clears the floor
  heel strike       Lafortune 1991 J Biomech (bone-mounted tibial accelerometer, walking:
                    resultant 2.7-3.7 g after foot strike; AP peak 1.26 g shod, 2.33 g barefoot)
Approximate rules (marked APPROX in the code) are stated as such and are meant to be replaced by
dataset regressions (e.g. Fukuchi 2018 WBDS) in v2.
"""
import os, json, urllib.request, xml.etree.ElementTree as ET
import numpy as np, pandas as pd, mujoco
from scipy.interpolate import CubicSpline, PchipInterpolator
from scipy.signal import butter, sosfiltfilt, sosfilt

G = 9.80665
FS = 1000                      # internal rate, Hz
BASE = ('https://raw.githubusercontent.com/opensim-org/opensim-models/master/'
        'Pipelines/Gait2354_Simbody/')
FILES = {'ik': 'OutputReference/subject01_walk1_ik.mot',
         'osim': 'OutputReference/subject01_simbody.osim',
         'grf': 'subject01_walk1_grf.mot'}
LEG_COORDS = ('hip_flexion', 'hip_adduction', 'hip_rotation', 'knee_angle')
PELVIS_ROT = ('pelvis_tilt', 'pelvis_list', 'pelvis_rotation')
NGRID = 1000                   # samples per stride when building a stride pattern


# =============================================================================
# small utilities
# =============================================================================
def fetch(key, folder='data'):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, os.path.basename(FILES[key]))
    if not os.path.exists(path):
        urllib.request.urlretrieve(BASE + FILES[key], path)
    return path


def fourier_fit(x, y, n, period=1.0):
    w = 2 * np.pi / period
    A = np.column_stack([np.ones_like(x)] + [f(k * w * x) for k in range(1, n + 1)
                                             for f in (np.cos, np.sin)])
    c, *_ = np.linalg.lstsq(A, y, rcond=None)
    return c


def fourier_eval(c, x, period=1.0, deriv=0):
    w = 2 * np.pi / period
    out = np.full_like(x, c[0] if deriv == 0 else 0.0, dtype=float)
    for k in range(1, (len(c) - 1) // 2 + 1):
        a, b, kw = c[2 * k - 1], c[2 * k], k * w
        cs, sn = np.cos(kw * x), np.sin(kw * x)
        if deriv == 0:
            out += a * cs + b * sn
        elif deriv == 1:
            out += kw * (-a * sn + b * cs)
        else:
            out += -kw ** 2 * (a * cs + b * sn)
    return out


def raised_cos(x):
    """0 -> 1 smooth ramp for x in [0, 1]."""
    x = np.clip(x, 0, 1)
    return 0.5 - 0.5 * np.cos(np.pi * x)


def periodic_bump(phi, start, end):
    """Smooth half-sine bump (0..1..0) on the periodic interval [start, end) of phase."""
    L = (end - start) % 1.0
    u = ((phi - start) % 1.0) / max(L, 1e-9)
    return np.where(u < 1.0, np.sin(np.pi * np.clip(u, 0, 1)) ** 2, 0.0)


# =============================================================================
# template from the public pattern
# =============================================================================
def read_osim(path):
    root = ET.parse(path).getroot()
    J = {j.get('name'): j for j in root.iter('CustomJoint')}
    out = {}
    for side in 'rl':
        hip = J['hip_' + side]
        out['hip_' + side] = np.array(hip.find('frames')[0].find('translation').text.split(), float)
        out['hip_axes_' + side] = [np.array(ta.find('axis').text.split(), float)
                                   for ta in hip.iter('TransformAxis')][:3]
        for ta in J['knee_' + side].iter('TransformAxis'):
            if ta.get('name') in ('translation1', 'translation2'):
                mf = ta.find('MultiplierFunction')
                sp = mf.find('function').find('SimmSpline')
                x = np.array(sp.find('x').text.split(), float)
                y = np.array(sp.find('y').text.split(), float) * float(mf.find('scale').text)
                out[f"knee_{ta.get('name')[-1]}_{side}"] = CubicSpline(x, y, bc_type='natural')
        out['ankle_' + side] = np.array(J['ankle_' + side].find('frames')[0]
                                        .find('translation').text.split(), float)
    return out


def gait_events(grf_path):
    g = pd.read_csv(grf_path, sep=r'\s+', skiprows=6)
    ev = {}
    for p in ('', '1_'):
        on = g[p + 'ground_force_vy'].values > 20
        side = 'r' if g[p + 'ground_force_pz'].values[on].mean() > 0 else 'l'
        e = np.flatnonzero(np.diff(on.astype(int))) + 1
        ev['hs_' + side] = g.time.values[e[on[e]]]
        ev['to_' + side] = g.time.values[e[~on[e]]]
    return ev


class Template:
    """One symmetric stride of healthy walking, as functions of gait phase (0 = heel strike)."""

    def __init__(self, n_harm=7, n_leg=10):
        ik = pd.read_csv(fetch('ik'), sep=r'\s+', skiprows=10)
        self.osim = read_osim(fetch('osim'))
        ev = gait_events(fetch('grf'))
        T = float(np.mean([np.diff(ev['hs_r']).mean(), np.diff(ev['hs_l']).mean()]))
        t0 = ik.time.values[0]
        hs_r = ev['hs_r'][ev['hs_r'] > t0][0]
        hs_l = ev['hs_l'][ev['hs_l'] > hs_r][0]
        to_r = ev['to_r'][ev['to_r'] > hs_r][0]
        to_l = ev['to_l'][ev['to_l'] > hs_l][0]
        to_l0 = ev['to_l'][ev['to_l'] > hs_r][0]
        to_r0 = ev['to_r'][ev['to_r'] > hs_l][0]
        self.T, self.v = T, None
        # raw (non-symmetric) fits as functions of phase relative to right heel strike
        x = ((ik.time.values - hs_r) / T)
        self.raw = {}
        for col in ik.columns[1:]:
            is_tr = col in ('pelvis_tx', 'pelvis_ty', 'pelvis_tz')
            y = ik[col].values if is_tr else np.radians(ik[col].values)
            self.raw[col] = fourier_fit(x, y, n_harm)
        self.phi_hs_l = (hs_l - hs_r) / T
        # events of a generic leg, relative to its own heel strike (mean of both legs)
        self.phi_to = 0.5 * ((to_r - hs_r) + (to_l - hs_l)) / T
        self.phi_cto = 0.5 * ((to_l0 - hs_r) + (to_r0 - hs_l)) / T     # contralateral toe-off
        # symmetric leg template on a grid
        g = np.linspace(0, 1, NGRID, endpoint=False)
        self.leg = {}
        for c in LEG_COORDS:
            r = fourier_eval(self.raw[c + '_r'], g)
            l = fourier_eval(self.raw[c + '_l'], g + self.phi_hs_l)
            self.leg[c] = fourier_fit(g, 0.5 * (r + l), n_leg)
        self.pelvis = {}
        for c, sgn in (('pelvis_tilt', 1), ('pelvis_list', -1), ('pelvis_rotation', -1)):
            a = fourier_eval(self.raw[c], g)
            b = fourier_eval(self.raw[c], g + 0.5)
            self.pelvis[c] = fourier_fit(g, 0.5 * (a + sgn * b), 10)
        self.grid = g


# =============================================================================
# MuJoCo model with scalable segments and sensor sites
# =============================================================================
REF = dict(femur=0.4552, tibia=0.4251, pelvis=1.0)   # template lengths (m) / scale


def site_xyaxes(yaw_deg=0.0, slope_deg=0.0, roll_deg=0.0):
    """Device axes in tibia coordinates (tibia: X anterior, Y superior, Z right).
    Nominal: x_dev = X, y_dev = -Z (left), z_dev = Y. Mounting errors rotate the device about
    its own axes: yaw about z_dev, posterior slope about y_dev, roll about x_dev."""
    R0 = np.array([[1, 0, 0], [0, 0, 1], [0, -1, 0]], float)      # columns = device axes
    def rot(axis, a):
        a = np.radians(a); c, s = np.cos(a), np.sin(a)
        if axis == 'x': return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
        if axis == 'y': return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    # posterior slope: the insert plane dips posteriorly, so z_dev tilts posteriorly -> rotation
    # about +y_dev (left) by -slope tilts z toward -x
    R = R0 @ rot('z', yaw_deg) @ rot('y', -slope_deg) @ rot('x', roll_deg)
    return R, ' '.join(f'{v:.6f}' for v in np.r_[R[:, 0], R[:, 1]])


def build_model(osim, femur=REF['femur'], tibia=REF['tibia'], pelvis=1.0,
                imu_side='r', imu_pos=(0.005, -0.025, 0.0), mount=(0.0, 0.0, 0.0),
                shin_frac=0.47):
    """Return (MjModel, info). Lengths in m; mount = (yaw, slope, roll) in deg."""
    kf = femur / REF['femur']
    _, xy_imu = site_xyaxes(*mount)
    _, xy_nom = site_xyaxes()
    legs = ''
    for s in 'rl':
        h = osim['hip_' + s] * pelvis
        ax = lambda v: ' '.join(f'{c:g}' for c in v)
        a = osim['hip_axes_' + s]
        imu = (f'<site name="imu" pos="{imu_pos[0]} {imu_pos[1]} {imu_pos[2]}" xyaxes="{xy_imu}"/>'
               if s == imu_side else '')
        legs += f"""
      <body name="femur_{s}" pos="{h[0]} {h[1]} {h[2]}">
        <joint name="hip_flexion_{s}" axis="{ax(a[0])}"/>
        <joint name="hip_adduction_{s}" axis="{ax(a[1])}"/>
        <joint name="hip_rotation_{s}" axis="{ax(a[2])}"/>
        <geom type="capsule" fromto="0 0 0 0 {-femur:.4f} 0" size="0.05" mass="9"/>
        <site name="thigh_{s}" pos="0.06 {-(femur - 0.05):.4f} 0" xyaxes="{xy_nom}"/>
        <body name="tibia_{s}">
          <joint name="knee_tx_{s}" type="slide" axis="1 0 0"/>
          <joint name="knee_ty_{s}" type="slide" axis="0 1 0"/>
          <joint name="knee_angle_{s}" axis="0 0 1"/>
          <geom type="capsule" fromto="0 0 0 0 {-tibia:.4f} 0" size="0.04" mass="3.5"/>
          {imu}
          <site name="shin_{s}" pos="0.025 {-shin_frac * tibia:.4f} 0" xyaxes="{xy_nom}"/>
          <site name="ankle_{s}" pos="0 {-tibia:.4f} 0"/>
        </body>
      </body>"""
    xml = f"""<mujoco model="kneil_v1">
  <option gravity="0 {-G} 0" timestep="{1 / FS}"/>
  <worldbody>
    <body name="pelvis">
      <joint name="pelvis_tx" type="slide" axis="1 0 0"/>
      <joint name="pelvis_ty" type="slide" axis="0 1 0"/>
      <joint name="pelvis_tz" type="slide" axis="0 0 1"/>
      <joint name="pelvis_tilt" axis="0 0 1"/>
      <joint name="pelvis_list" axis="1 0 0"/>
      <joint name="pelvis_rotation" axis="0 1 0"/>
      <geom type="sphere" size="0.09" mass="11"/>{legs}
    </body>
  </worldbody>
  <sensor>
    <accelerometer name="acc_imu" site="imu"/>
    <gyro name="gyr_imu" site="imu"/>
    <accelerometer name="acc_shin_r" site="shin_r"/>
    <gyro name="gyr_shin_r" site="shin_r"/>
    <accelerometer name="acc_shin_l" site="shin_l"/>
    <gyro name="gyr_shin_l" site="shin_l"/>
    <accelerometer name="acc_thigh_r" site="thigh_r"/>
    <accelerometer name="acc_thigh_l" site="thigh_l"/>
  </sensor>
</mujoco>"""
    m = mujoco.MjModel.from_xml_string(xml)
    knee = {k: (lambda th, d=0, f=osim[k]: kf * f(th, d)) for k in osim if k.startswith('knee_')}
    return m, dict(knee=knee, femur=femur, tibia=tibia, pelvis=pelvis, kf=kf, imu_side=imu_side)


def qidx(m):
    return {m.joint(i).name: m.jnt_qposadr[i] for i in range(m.njnt)}


def set_knee_translations(Q, idx, info):
    """Fill the coupled knee translation coordinates from the knee angles (rows = samples)."""
    for s in 'rl':
        th = Q[:, idx['knee_angle_' + s]]
        Q[:, idx['knee_tx_' + s]] = info['knee']['knee_1_' + s](th)
        Q[:, idx['knee_ty_' + s]] = info['knee']['knee_2_' + s](th)
    return Q


def ankle_vectors(m, Q):
    """Ankle centre positions (world axes) for each row of Q; pelvis translation as given."""
    d = mujoco.MjData(m)
    ids = [m.site('ankle_r').id, m.site('ankle_l').id]
    out = np.zeros((len(Q), 2, 3))
    for i, q in enumerate(Q):
        d.qpos[:] = q
        mujoco.mj_kinematics(m, d)
        out[i] = d.site_xpos[ids]
    return out


# =============================================================================
# one stride: warped template + modifications + pelvis from stance-foot constraints
# =============================================================================
def stance_profiles(psi, heel_rise=1.0):
    """Ankle-centre height and forward roll of the stance foot vs stance progress psi (0..1),
    shaped to the template (heel rocker 2.4 cm / 3.3 cm, heel rise to 12.8 cm at toe-off)."""
    heel = 1 - raised_cos(np.clip(psi, 0, 1) / 0.2)
    h = 0.024 * heel + heel_rise * 0.128 * np.clip((psi - 0.6) / 0.4, 0, 1) ** 2.2
    d = -0.033 * heel + heel_rise * 0.123 * np.clip((psi - 0.8) / 0.2, 0, 1) ** 2
    return h, d


def stance_weight(phi, to, cto, chs):
    """Share of body support on a leg (phase from its own heel strike)."""
    w = np.zeros_like(phi)
    a = phi < cto
    w[a] = raised_cos(phi[a] / cto)
    b = (phi >= cto) & (phi < chs)
    w[b] = 1.0
    c = (phi >= chs) & (phi < to)
    w[c] = 1 - raised_cos((phi[c] - chs) / (to - chs))
    return w


def phase_warp(TP, to, cto, chs):
    """Smooth periodic map from a leg's own phase to template phase (events aligned)."""
    x = np.array([to - 1, 0, cto, chs, to, 1, 1 + cto])
    y = np.array([TP.phi_to - 1, 0, TP.phi_cto, 0.5, TP.phi_to, 1, 1 + TP.phi_cto])
    return PchipInterpolator(x, y)


def knee_features(TP):
    g = TP.grid
    f = -np.degrees(fourier_eval(TP.leg['knee_angle'], g))     # flexion positive
    st_min = f[(g > 0.25) & (g < 0.55)].min()
    lr = f[g < 0.3].max()
    peak = f[g > 0.6].max()
    return f, dict(min=st_min, lr=lr, peak=peak)


def gait_params(subject, cond, rng):
    """Map a virtual subject and a condition to gait parameters (see module docstring)."""
    p = {}
    # comfortable speed: Bohannon & Andrews 2011 decade means (m/s) by sex
    ages = np.array([25, 35, 45, 55, 65, 75, 88])
    vm = np.array([1.358, 1.433, 1.434, 1.433, 1.339, 1.262, 0.968])
    vf = np.array([1.341, 1.337, 1.390, 1.401, 1.300, 1.239, 0.943])
    v_norm = np.interp(subject['age'], ages, vm if subject['sex'] == 'M' else vf)
    v = v_norm * subject.get('speed_factor', 1.0)
    s = cond.get('severity', 0.0)
    v *= (1 - 0.5 * s)                                  # APPROX: s = 1 -> half normal speed
    if cond.get('locked'):
        v *= 0.6                                        # APPROX: braced/locked walking ~0.6x
    if cond.get('aid') in ('crutches', 'walker'):
        v *= 0.8                                        # APPROX
    p['v'] = float(np.clip(v, 0.25, 1.9))
    # stride length from dimensionless speed (Alexander 1984), leg length ~ 0.53 H
    L_leg = 0.53 * subject['height']
    Fr = p['v'] ** 2 / (G * L_leg)
    # APPROX: coefficient 2.55 (Alexander's form) matches the template subject and adult norms
    p['stride'] = float(2.55 * Fr ** 0.3 * L_leg * subject.get('stride_factor', 1.0))
    p['T'] = p['stride'] / p['v']
    # timing: stance fraction rises at slow speed (APPROX slope), antalgic asymmetry with s
    base_to = float(np.clip(0.639 + 0.06 * (1.08 - p['v']), 0.58, 0.72))
    asym = 0.03 * s + (0.02 if cond.get('locked') else 0.0)
    p['to_a'] = base_to - asym                         # affected (device) leg: shorter stance
    p['to_b'] = base_to + asym
    p['hs_b'] = 0.5 - asym                             # shorter affected single support
    # knee targets (flexion, deg): typical adult values at 1.2 m/s (loading response ~17 deg,
    # swing peak ~62 deg) with subject variation; speed slopes APPROX after Lelas 2003 trends.
    # (The template subject's own 20/69 deg are on the high side; HuGaDB gyro data gave ~54 deg
    # knee range of motion.)
    p['lr_target'] = 17.0 + 12.0 * (p['v'] - 1.2) + subject.get('lr_dev', 0.0)
    p['peak_target'] = 62.0 + 8.0 * (p['v'] - 1.2) + subject.get('peak_dev', 0.0)
    p['severity'] = s
    p['locked'] = bool(cond.get('locked', False))
    p['lock_angle'] = cond.get('lock_angle', 5.0)
    p['ext_deficit'] = 8.0 * s                         # Debbi 2013: ~8 deg at 2 months
    p['swing_scale'] = 1 - 0.5 * s                     # s = 1 -> ~41 deg peak (Debbi: 41 deg)
    p['lr_scale'] = 1 - 0.6 * s                        # stiff loading response
    p['heel_rise_a'] = 1 - 0.5 * s - (0.3 if p['locked'] else 0.0)
    p['clear_req'] = 0.85 * 0.018 * subject['height'] / 1.79   # 85 % of template clearance
    return p


def build_stride(TP, m, info, p, side='r', solve=True, ngrid=NGRID, fixed=None):
    """Return dict with one stride (phase grid from the device-leg heel strike)."""
    A, B = side, ('l' if side == 'r' else 'r')
    g = np.linspace(0, 1, ngrid, endpoint=False)
    idx = qidx(m)
    to_a, to_b, hs_b = p['to_a'], p['to_b'], p['hs_b']
    cto_a = (hs_b + to_b) % 1.0                        # B toe-off, in A phase
    events = dict(hs_a=0.0, to_a=to_a, hs_b=hs_b, to_b=cto_a)
    phiA, phiB = g, (g - hs_b) % 1.0
    warpA = phase_warp(TP, to_a, cto_a, hs_b)
    warpB = phase_warp(TP, to_b, (to_a - hs_b) % 1.0, (1 - hs_b) % 1.0)
    fk, kfeat = knee_features(TP)

    def leg_angles(phi_leg, warp, affected, a_hip, comp):
        tp = warp(phi_leg) % 1.0
        out = {c: fourier_eval(TP.leg[c], tp) for c in LEG_COORDS}
        # hip sagittal amplitude (solved for stride length)
        h = out['hip_flexion']; hm = h.mean()
        out['hip_flexion'] = hm + a_hip * (h - hm)
        # knee
        f = -np.degrees(out['knee_angle'])
        sev = p['severity'] if affected else 0.0
        mn = kfeat['min'] + (p['ext_deficit'] if affected else 0.0)
        lr_t = p['lr_target']
        pk_t = p['peak_target']
        if affected:
            lr_t = mn + (lr_t - mn) * p['lr_scale']
            pk_t = mn + (pk_t - kfeat['min']) * p['swing_scale']
        w_st = max((lr_t - mn) / (kfeat['lr'] - kfeat['min']), 0.05)
        w_sw = max((pk_t - mn) / (kfeat['peak'] - kfeat['min']), 0.05)
        S = raised_cos((tp - 0.45) / 0.2) * (1 - raised_cos((tp - 0.9) / 0.1))
        wgt = w_st + (w_sw - w_st) * S
        fn = mn + (f - kfeat['min']) * wgt
        if affected and p['locked']:
            fn = p['lock_angle'] + 0.03 * (f - kfeat['min'])
        out['knee_angle'] = -np.radians(fn)
        # compensations during this leg's swing (hip abduction = negative adduction)
        if comp is not None:
            out['hip_adduction'] = out['hip_adduction'] - np.radians(comp['abd']) * comp['bump']
        return out

    # compensation windows of the device leg: hiking/abduction from pre-swing to late swing
    # (flat top), vaulting of the other leg around mid-swing (its mid-stance)
    u = ((g - (to_a - 0.08)) % 1.0)
    span = (0.98 - (to_a - 0.08))
    bumpA = np.where(u < span, raised_cos(u / 0.10) * (1 - raised_cos((u - (span - 0.10)) / 0.10)), 0.0)
    bumpV = periodic_bump(g, to_a + 0.05, 0.92)

    def assemble(a_hip, k_comp):
        comp = dict(abd=4.0 * k_comp, bump=bumpA) if k_comp > 0 else None
        la = leg_angles(phiA, warpA, True, a_hip, comp)
        lb = leg_angles(phiB, warpB, False, a_hip, None)
        Q = np.zeros((len(g), m.nq))
        for c in LEG_COORDS:
            Q[:, idx[f'{c}_{A}']] = la[c]
            Q[:, idx[f'{c}_{B}']] = lb[c]
        # pelvis rotations follow the device-leg phase (right-leg template phase)
        tpA = warpA(phiA) % 1.0
        sgn = 1.0 if A == 'r' else -1.0                    # list/rotation flip for a left reference
        Q[:, idx['pelvis_tilt']] = fourier_eval(TP.pelvis['pelvis_tilt'], tpA)
        Q[:, idx['pelvis_list']] = sgn * fourier_eval(TP.pelvis['pelvis_list'], tpA)
        Q[:, idx['pelvis_rotation']] = sgn * fourier_eval(TP.pelvis['pelvis_rotation'], tpA)
        if k_comp > 0:                                     # hip hiking on the swing side
            hike = np.radians(3.0 * k_comp) * bumpA
            Q[:, idx['pelvis_list']] += (-hike if A == 'r' else hike)
        Q = set_knee_translations(Q, idx, info)
        Av = ankle_vectors(m, Q)                           # pelvis translation = 0 here
        ia, ib = (0, 1) if A == 'r' else (1, 0)
        wa = stance_weight(phiA, to_a, cto_a, hs_b)
        wb = stance_weight(phiB, to_b, (to_a - hs_b) % 1.0, (1 - hs_b) % 1.0)
        ha, da = stance_profiles(phiA / to_a, p['heel_rise_a'])
        hb, db = stance_profiles(phiB / to_b, 1.0)
        if k_comp > 0:                                     # vaulting: contralateral heel rise
            hb = hb + 0.02 * k_comp * bumpV
        ty = wa * (ha - Av[:, ia, 1]) + wb * (hb - Av[:, ib, 1])
        T = p['T']
        dA = (np.roll(Av, -1, 0) - np.roll(Av, 1, 0)) / (2 / len(g)) / T     # d/dt
        dda = (np.roll(da, -1) - np.roll(da, 1)) / (2 / len(g)) / T
        ddb = (np.roll(db, -1) - np.roll(db, 1)) / (2 / len(g)) / T
        vx = wa * (dda * (phiA < to_a) - dA[:, ia, 0]) + wb * (ddb * (phiB < to_b) - dA[:, ib, 0])
        vz = -(wa * dA[:, ia, 2] + wb * dA[:, ib, 2])
        vz -= vz.mean()
        L = vx.mean() * T                                  # stride length
        x = np.cumsum(vx) * T / len(g); x -= x[0]
        z = np.cumsum(vz) * T / len(g)
        Q[:, idx['pelvis_tx']] = x
        Q[:, idx['pelvis_ty']] = ty
        Q[:, idx['pelvis_tz']] = z - z.mean()
        swing_a = (phiA > to_a + 0.02) & (phiA < 0.92)
        clear = (ty + Av[:, ia, 1])[swing_a].min()          # swing ankle height (stance flat = 0)
        return Q, L, clear, dict(wa=wa, wb=wb, swing_h=(ty + Av[:, ia, 1]), swing_mask=swing_a)

    a_hip, k_comp = (1.0, 0.0) if fixed is None else fixed
    if solve:
        # required swing clearance = healthy template clearance (computed once per call)
        for _ in range(3):
            # stride length: secant on hip amplitude
            a0, a1 = 0.8 * a_hip, 1.2 * a_hip
            L0 = assemble(a0, k_comp)[1]; L1 = assemble(a1, k_comp)[1]
            for _ in range(6):
                if abs(L1 - L0) < 1e-9:
                    break
                a2 = a1 + (p['stride'] - L1) * (a1 - a0) / (L1 - L0)
                a2 = float(np.clip(a2, 0.25, 1.8))
                a0, L0, a1 = a1, L1, a2
                L1 = assemble(a1, k_comp)[1]
                if abs(L1 - p['stride']) < 0.002:
                    break
            a_hip = a1
            # clearance: raise compensation until the swing ankle clears like healthy gait
            c = assemble(a_hip, k_comp)[2]
            if c >= p.get('clear_req', 0.0) - 1e-4:
                break
            lo, hi = k_comp, 6.0
            for _ in range(12):
                mid = 0.5 * (lo + hi)
                if assemble(a_hip, mid)[2] >= p['clear_req']:
                    hi = mid
                else:
                    lo = mid
            k_comp = hi
    Q, L, clear, w = assemble(a_hip, k_comp)
    return dict(Q=Q, grid=g, L=L, clear=clear, a_hip=a_hip, k_comp=k_comp, events=events,
                side=A, T=p['T'], weights=w)


# =============================================================================
# trial: variability, replay, impacts, sensor model
# =============================================================================
H_REF = 1.79      # stature that matches the template segment lengths (Winter ratios)


def smooth_noise(t, step, sd, rng):
    """Smooth random signal: N(0, sd) knots every `step` seconds, cubic interpolation."""
    kn = np.arange(t[0] - step, t[-1] + 2 * step, step)
    return CubicSpline(kn, rng.normal(0, sd, len(kn)))(t)


def impact_kernel(f_n, zeta, fs=FS, dur=0.12):
    tt = np.arange(0, dur, 1 / fs)
    wn = 2 * np.pi * f_n
    wd = wn * np.sqrt(1 - zeta ** 2)
    k = np.exp(-zeta * wn * tt) * np.sin(wd * tt)
    return k / np.abs(k).max()


def sample_subject(rng, kind='patient'):
    """Virtual subject. Patients: typical TKA-revision ages; young: HuGaDB-like adults."""
    if kind == 'young':
        sex = 'M' if rng.random() < 0.78 else 'F'
        age = rng.uniform(19, 30)
    else:
        sex = 'M' if rng.random() < 0.5 else 'F'
        age = float(np.clip(rng.normal(68, 9), 45, 90))
    h_mean, h_sd = (1.76, 0.07) if sex == 'M' else (1.63, 0.065)
    if kind == 'young':
        h_mean += 0.03
    height = float(np.clip(rng.normal(h_mean, h_sd), 1.45, 2.05))
    bmi = float(np.clip(rng.normal(30 if kind == 'patient' else 23, 5 if kind == 'patient' else 3), 18, 48))
    return dict(sex=sex, age=round(age, 1), height=round(height, 3), bmi=round(bmi, 1),
                mass=round(bmi * height ** 2, 1), side=('r' if rng.random() < 0.5 else 'l'),
                speed_factor=float(np.clip(rng.normal(1.0, 0.10), 0.7, 1.3)),
                stride_factor=float(np.clip(rng.normal(1.0, 0.05), 0.85, 1.15)),
                femur_dev=float(rng.normal(0, 0.03)), tibia_dev=float(rng.normal(0, 0.03)),
                lr_dev=float(rng.normal(0, 4.0)), peak_dev=float(rng.normal(0, 4.0)))


def sample_sensor(rng):
    """Implant mounting and accelerometer errors (APPROX generic low-power MEMS values)."""
    return dict(yaw=float(np.clip(rng.normal(0, 4), -10, 10)), slope=float(rng.uniform(0, 7)),
                roll=float(rng.normal(0, 2)),
                pos=(0.005 + rng.normal(0, 0.004), -0.025 + rng.normal(0, 0.005), rng.normal(0, 0.004)),
                bias=rng.normal(0, 0.02, 3).tolist(), scale=rng.normal(0, 0.01, 3).tolist(),
                cross=rng.normal(0, 0.005, 3).tolist(), noise_ug=150.0,
                impact_f=float(rng.uniform(15, 30)), impact_zeta=float(rng.uniform(0.3, 0.5)))


def make_trial(TP, subj, cond, sens, duration, rng, footwear='shoes', shin=False, margin=2.0, f_bw=4.0, jitter_deg=0.0):
    k = subj['height'] / H_REF
    m, info = build_model(TP.osim, femur=REF['femur'] * k * (1 + subj['femur_dev']),
                          tibia=REF['tibia'] * k * (1 + subj['tibia_dev']), pelvis=k,
                          imu_side=subj['side'], imu_pos=sens['pos'],
                          mount=(sens['yaw'], sens['slope'], sens['roll']))
    idx = qidx(m)
    p = gait_params(subj, cond, rng)
    S = build_stride(TP, m, info, p, subj['side'])
    T, g = p['T'], S['grid']
    # kinematic bandwidth: 4 Hz, calibrated on HuGaDB participants 01/03/09/10 (thigh and shin
    # signal ranges), checked on participants 14/15/16/17 (see validate.py)
    nh = int(max(3, np.floor(f_bw * T)))
    coef = {}
    for name, a in idx.items():
        if name.startswith('knee_t'):
            continue
        y = S['Q'][:, a]
        if name == 'pelvis_tx':
            y = y - S['L'] * g
        coef[name] = fourier_fit(g, y, nh)
    # time base and variability
    t = np.arange(-margin, duration + margin, 1 / FS)
    sev = cond.get('severity', 0.0)
    cv_T = 0.02 + 0.02 * sev + (0.01 if cond.get('locked') else 0.0)   # APPROX stride-time CV
    cv_A = 0.03 + 0.02 * sev
    r = smooth_noise(t, T, 1.6 * cv_T, rng)                  # 1.6: smoothing loss, gives cv_T
    phi = np.cumsum(1.0 / (T * (1 + r))) / FS
    phi += rng.uniform(0, 1) - phi[np.argmin(abs(t))]       # random phase at t = 0
    eta = {grp: smooth_noise(t, T, cv_A, rng) for grp in ('hip', 'knee', 'other')}
    Q = np.zeros((len(t), m.nq))
    ph = phi % 1.0
    for name, c in coef.items():
        y = fourier_eval(c, ph)
        grp = 'hip' if name.startswith('hip_flexion') else 'knee' if name.startswith('knee') else 'other'
        y = c[0] + (y - c[0]) * (1 + eta[grp])
        Q[:, idx[name]] = y
    Q[:, idx['pelvis_tx']] += np.cumsum(S['L'] * (1 + eta['hip']) * np.gradient(phi))
    if jitter_deg > 0:   # stride-to-stride irregularity: 3-8 Hz band-limited hip/knee jitter (calibrated, see tstr)
        sos_j = butter(4, [3.0, 8.0], btype='band', fs=FS, output='sos')
        for nm_ in ('hip_flexion_r', 'hip_flexion_l', 'knee_angle_r', 'knee_angle_l'):
            w_ = sosfiltfilt(sos_j, rng.normal(size=len(t))); Q[:, idx[nm_]] += np.radians(jitter_deg) * w_ / (w_.std() + 1e-12)
    Q = set_knee_translations(Q, idx, info)
    QD = np.gradient(Q, 1 / FS, axis=0)
    QDD = np.gradient(QD, 1 / FS, axis=0)
    # replay
    d = mujoco.MjData(m)
    names = ['acc_imu', 'gyr_imu'] + (['acc_shin_r', 'gyr_shin_r', 'acc_shin_l', 'gyr_shin_l',
                                        'acc_thigh_r', 'acc_thigh_l'] if shin else [])
    adr = {n: m.sensor(n).adr[0] for n in names}
    out = {n: np.zeros((len(t), 3)) for n in names}
    tib = m.body('tibia_' + subj['side']).id
    tilt = np.zeros(len(t))
    for i in range(len(t)):
        d.qpos[:], d.qvel[:], d.qacc[:] = Q[i], QD[i], QDD[i]
        mujoco.mj_inverse(m, d)
        for n in names:
            out[n][i] = d.sensordata[adr[n]:adr[n] + 3]
        ax = d.xmat[tib].reshape(3, 3)[:, 1]                    # tibia long axis (superior)
        tilt[i] = np.degrees(np.arctan2(ax[0], ax[1]))          # + = top of tibia forward
    for n in names:
        out[n] = out[n] / (G if n.startswith('acc') else np.pi / 180)   # g and deg/s
    # gait events of the device leg (and the other leg)
    def crossings(x, level):
        return np.flatnonzero((np.floor(x[1:] - level) - np.floor(x[:-1] - level)) > 0) + 1
    ev = dict(hs=crossings(phi, 0.0), to=crossings(phi, S['events']['to_a']),
              hs_other=crossings(phi, S['events']['hs_b']), to_other=crossings(phi, S['events']['to_b']))
    # heel-strike transients (tibia axes: axial +, posterior first, small ML), Lafortune 1991 scale
    v = p['v']
    fw = dict(shoes=1.0, slippers=0.8, barefoot=1.5)[footwear]
    aid = 0.7 if cond.get('aid') in ('crutches', 'walker') else 1.0
    gain = (v / 1.3) ** 1.5 * fw * aid * (1 - 0.4 * sev)
    ker = impact_kernel(sens['impact_f'], sens['impact_zeta'])
    imp = np.zeros((len(t), 3))                                 # in tibia anatomical axes x,y(left),z
    for i0 in ev['hs']:
        a = gain * rng.lognormal(0, 0.15)
        ml = rng.choice([-1, 1]) * 0.3
        seg = slice(i0, min(i0 + len(ker), len(t)))
        n = seg.stop - seg.start
        imp[seg] += np.outer(ker[:n], [-0.8 * a, ml * a, 1.6 * a])
    R, _ = site_xyaxes(sens['yaw'], sens['slope'], sens['roll'])
    Rn, _ = site_xyaxes()
    imp_dev = imp @ (np.linalg.inv(R) @ Rn).T                # nominal device axes -> mounted axes
    acc_true = out['acc_imu'] + imp_dev
    res = dict(t=t, acc_true=acc_true, gyr=out['gyr_imu'], phi=phi, ev=ev, p=p, S=S, Q=Q,
               knee=-np.degrees(Q[:, idx['knee_angle_' + subj['side']]]), tilt=tilt, model=m,
               imp_gain=gain, margin=margin)
    if shin:
        res['shin'] = {s: (out['acc_shin_' + s], out['gyr_shin_' + s]) for s in 'rl'}
        res['thigh'] = {s: out['acc_thigh_' + s] for s in 'rl'}
        res['imp_tibia'] = imp
    return res


def device_output(res, sens, rng, odr=100, range_g=8.0, bits=16):
    """Accelerometer chain: errors, noise, anti-alias filter, decimation, quantization, clipping."""
    a = res['acc_true'].copy()
    C = np.eye(3) + np.diag(sens['scale'])
    c = sens['cross']
    C[0, 1] += c[0]; C[1, 0] += c[0]; C[0, 2] += c[1]; C[2, 0] += c[1]; C[1, 2] += c[2]; C[2, 1] += c[2]
    a = a @ C.T + np.asarray(sens['bias'])
    a += rng.normal(0, sens['noise_ug'] * 1e-6 * np.sqrt(FS / 2), a.shape)
    sos = butter(4, 0.4 * odr, fs=FS, output='sos')
    a = sosfilt(sos, a, axis=0)
    step = FS // odr
    keep = (res['t'] >= 0) & (res['t'] < res['t'][-1] - res['margin'] + 1e-9)
    i = np.flatnonzero(keep)[::step]
    lsb = 2 * range_g / 2 ** bits
    a = np.round(np.clip(a[i], -range_g, range_g - lsb) / lsb) * lsb
    return i, a


# =============================================================================
# validation against HuGaDB shin recordings (Chereshnev & Kertesz-Farkas 2017)
# =============================================================================
HUGADB_FS = 56.35         # Hz, mean rate reported by the authors (SD 3.2, no time stamps)


def load_hugadb(path):
    """Shin signals rotated to anatomical axes (x anterior, y left, z superior), in g and deg/s.
    z from standing gravity; y from the main gyroscope axis in walking (unclipped samples);
    y sign from mid-swing (shank angular velocity points to the subject's right)."""
    d = pd.read_csv(path, sep='\t', comment='#')
    act = d['act'].values
    out = {}
    for leg, side in (('rs', 'r'), ('ls', 'l')):
        ra = d[[f'acc_{leg}_{a}' for a in 'xyz']].values
        rg = d[[f'gyro_{leg}_{a}' for a in 'xyz']].values
        acc, gyr = ra / 16384.0, rg * 2000 / 32768
        st, wk = act == 8, act == 1
        if st.sum() < 30 or wk.sum() < 200:
            continue
        z = acc[st].mean(0); z /= np.linalg.norm(z)
        ok = wk & (np.abs(rg) < 32767).all(1)
        Gm = gyr[ok] - gyr[ok].mean(0)
        y = np.linalg.eigh(Gm.T @ Gm)[1][:, -1]
        y -= (y @ z) * z; y /= np.linalg.norm(y)
        om = gyr[wk] @ y
        big = np.abs(om) > np.percentile(np.abs(om), 97)
        if om[big].mean() > 0:
            y = -y
        x = np.cross(y, z)
        Rm = np.vstack([x, y, z])
        out[side] = dict(acc=acc @ Rm.T, gyr=gyr @ Rm.T, act=act,
                         gyro_amplified=bool(np.percentile(np.abs(om), 97) > 1000))
    return out


def strides_midswing(acc, gyr, fs, mask=None, tmin=0.75, tmax=1.7, npts=101):
    """Cut strides between mid-swing peaks of the sagittal shank angular velocity."""
    from scipy.signal import find_peaks
    om = -gyr[:, 1]
    pk, _ = find_peaks(om, height=0.5 * np.percentile(om, 99), distance=int(0.6 * fs))
    S, Ts = [], []
    for a, b in zip(pk[:-1], pk[1:]):
        dur = (b - a) / fs
        if not (tmin <= dur <= tmax):
            continue
        if mask is not None and not mask[a:b + 1].all():
            continue
        u = np.linspace(a, b, npts)
        S.append(np.column_stack([np.interp(u, np.arange(len(acc)), acc[:, k]) for k in range(3)]))
        Ts.append(dur)
    return np.array(S), np.array(Ts)


def skin_hugadb_chain(t, acc, fs_out=HUGADB_FS, f_skin=20.0, z_skin=0.3, rng=None):
    """Mid-shank skin mounting (2nd-order resonance, APPROX) + MPU-type sampling (+-2 g)."""
    from scipy.signal import lti, lsim, cont2discrete, dlsim
    wn = 2 * np.pi * f_skin
    num, den = [wn ** 2], [1, 2 * z_skin * wn, wn ** 2]
    bd, ad, _ = cont2discrete((num, den), 1 / FS, method='bilinear')
    from scipy.signal import lfilter
    a = lfilter(bd.ravel(), ad, acc - acc[0], axis=0) + acc[0]
    sos = butter(4, 25.0, fs=FS, output='sos')
    a = sosfilt(sos, a, axis=0)
    tk = np.arange(t[0], t[-1], 1 / fs_out)
    a = np.column_stack([np.interp(tk, t, a[:, k]) for k in range(3)])
    a = np.round(np.clip(a, -2.0, 2.0 - 1 / 16384) * 16384) / 16384
    return tk, a
''')
print('wrote kneil_v1.py')

In [ ]:
#@title Write `camargo_replay.py` — replay of the Camargo joint angles
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/camargo_replay.py', 'w').write(r'''"""
Camargo et al. (2021) open dataset -> KNEIL MuJoCo replay.

Drives the KNEIL MuJoCo model (kneil_v1.build_model, gait2354 coordinate set, scaled to the subject's
height) with the dataset's OpenSim inverse kinematics, and returns specific force and angular velocity at
  - 'shin_r': a virtual sensor where Camargo's shank IMU sat (3/4 down the right shank, anterior surface)
  - 'imu'   : the KNEIL implant site in the right tibial insert (device frame: x anterior, y left, z superior)
Also: frame alignment of a real IMU to the simulated segment frame (Kabsch on the gyroscope), lever-arm
estimation, kinematic activity labels and foot-contact events.

Data layout accepted: <root>/<subject>/<mode>/{ik,imu}/<trial>.mat   (e.g. ab06/stair/ik/stair_1_l_01_01.mat)
                  or: <root>/<SUBJECT>/<date>/<mode>/{ik,imu}/<trial>.mat   (original dataset layout)
MATLAB tables are read with mat-io (pip install mat-io).
"""
import glob, os
import numpy as np
import mujoco
from scipy.interpolate import CubicSpline
from scipy.signal import butter, sosfiltfilt
import kneil_v1 as K

KDIR = os.path.dirname(os.path.abspath(__file__))
# Camargo et al. 2021, Table 1 (height in m)
HEIGHT = dict(ab06=1.80, ab07=1.65, ab08=1.74, ab09=1.63, ab10=1.75, ab11=1.75, ab12=1.74, ab13=1.73, ab14=1.52,
              ab15=1.78, ab16=1.65, ab17=1.68, ab18=1.80, ab19=1.70, ab20=1.71, ab21=1.57, ab23=1.80, ab24=1.73,
              ab25=1.63, ab27=1.70, ab28=1.69, ab30=1.77)
FS_DATA = 200.0
TRANSLATIONS = ('pelvis_tx', 'pelvis_ty', 'pelvis_tz')
_OSIM = None


def load_table(path):
    import matio
    return matio.load_from_mat(path)['data']


def find_trials(root, subj, mode):
    """Trial names that have both ik and imu tables, in either directory layout."""
    hits = sorted(glob.glob(os.path.join(root, subj, mode, 'ik', '*.mat')) +
                  glob.glob(os.path.join(root, subj.upper(), '*', mode, 'ik', '*.mat')))
    out = []
    for p in hits:
        q = p.replace(os.sep + 'ik' + os.sep, os.sep + 'imu' + os.sep)
        if os.path.exists(q):
            out.append((os.path.splitext(os.path.basename(p))[0], p, q))
    return out


def model_for(subj, side='r', shank_frac=0.75, implant_pos=(0.005, -0.025, 0.0), mount=(0.0, 0.0, 0.0)):
    global _OSIM
    if _OSIM is None:
        _OSIM = K.read_osim(K.fetch('osim', folder=os.path.join(KDIR, 'data')))
    k = HEIGHT[subj.lower()] / K.H_REF
    return K.build_model(_OSIM, femur=K.REF['femur'] * k, tibia=K.REF['tibia'] * k, pelvis=k, imu_side=side,
                         imu_pos=implant_pos, mount=mount, shin_frac=shank_frac)


def ik_to_Q(m, info, ik, fs_out=FS_DATA, f_lp=12.0):
    """IK table -> generalized coordinates (radians, metres), low-passed and resampled to fs_out."""
    idx = K.qidx(m)
    t = ik['Header'].values.astype(float)
    t = t - t[0]
    t_out = np.arange(0.0, t[-1] + 1e-9, 1.0 / fs_out)
    sos = butter(4, f_lp, fs=FS_DATA, output='sos')
    Q = np.zeros((len(t_out), m.nq))
    for name, a in idx.items():
        if name.startswith('knee_t') or name not in ik.columns:
            continue
        y = ik[name].values.astype(float)
        if name not in TRANSLATIONS:
            y = np.unwrap(np.radians(y))            # turns can wrap pelvis_rotation at +-180 deg
        y = sosfiltfilt(sos, y)
        Q[:, a] = y if len(t_out) == len(t) else CubicSpline(t, y)(t_out)
    return t_out, K.set_knee_translations(Q, idx, info)


def replay(m, Q, fs, sensors=('acc_imu', 'gyr_imu', 'acc_shin_r', 'gyr_shin_r')):
    """Kinematic replay with mj_inverse. Returns accelerometers in g, gyros in rad/s (site frames),
    'grav_<acc sensor>' = the static (gravity) part of that accelerometer in its frame (g; |grav| = 1),
    the right ankle position (world) and the right-tibia tilt (deg, + = top of tibia forward)."""
    QD = np.gradient(Q, 1.0 / fs, axis=0)
    QDD = np.gradient(QD, 1.0 / fs, axis=0)
    d = mujoco.MjData(m)
    adr = {n: m.sensor(n).adr[0] for n in sensors}
    site = {n: int(m.sensor_objid[m.sensor(n).id]) for n in sensors if n.startswith('acc')}
    out = {n: np.zeros((len(Q), 3)) for n in sensors}
    grav = {n: np.zeros((len(Q), 3)) for n in site}
    ank, tib, pel = m.site('ankle_r').id, m.body('tibia_r').id, m.body('pelvis').id
    ankle, tilt = np.zeros((len(Q), 3)), np.zeros(len(Q))
    for i in range(len(Q)):
        d.qpos[:], d.qvel[:], d.qacc[:] = Q[i], QD[i], QDD[i]
        mujoco.mj_inverse(m, d)
        for n in sensors:
            out[n][i] = d.sensordata[adr[n]:adr[n] + 3]
        for n, sid in site.items():
            grav[n][i] = d.site_xmat[sid].reshape(3, 3)[1, :]          # R^T (0, 1, 0): world 'up' in the site frame
        ankle[i] = d.site_xpos[ank]
        ax = d.xmat[tib].reshape(3, 3)[:, 1]                     # tibia long axis (superior), world
        fw = d.xmat[pel].reshape(3, 3)[:, 0]                     # pelvis anterior axis = heading
        fw = np.array([fw[0], 0.0, fw[2]]) / (np.hypot(fw[0], fw[2]) + 1e-12)
        tilt[i] = np.degrees(np.arctan2(ax @ fw, ax[1]))         # sagittal tilt in the heading frame
    for n in sensors:
        if n.startswith('acc'):
            out[n] /= K.G
    out.update({f'grav_{n}': g for n, g in grav.items()})
    out.update(ankle=ankle, tilt=tilt, QD=QD)
    return out


def scale_dynamic(acc, grav, k):
    """Scale only the movement part of a specific-force signal: k * (acc - grav) + grav (all in g)."""
    return grav + k * (acc - grav)


def draw_person(rng, calib=None):
    """Per-person randomisation from the 22-subject Camargo comparison (Nov 2026):
    impact speed exponent ~ N(1.35, 0.45) clipped to [0.5, 2.1] (panel c: median 1.34, range 0.60-1.95);
    movement-amplitude factor ~ U(0.82, 1.10) (panel b: sim/real amplitude 0.94-1.24);
    stair impact gains relative to walking at 1.3 m/s: log-normal from the real-IMU calibration (calib), else the
    earlier assumptions 0.5 (ascent) and 1.3 (descent)."""
    p = dict(impact_exponent=float(np.clip(rng.normal(1.35, 0.45), 0.5, 2.1)), amplitude=float(rng.uniform(0.82, 1.10)))
    if calib and calib.get('up') and calib.get('down'):
        for k, key in (('up', 'stair_up_gain'), ('down', 'stair_down_gain')):
            p[key] = float(np.exp(rng.normal(calib[k]['mu'], calib[k]['sd'])))
    else:
        p.update(stair_up_gain=0.5, stair_down_gain=1.3)
    return p


def lowpass_causal(x, fc, fs, order=2):
    """Causal Butterworth low-pass (what a device filter would do), along axis 0."""
    from scipy.signal import sosfilt
    return sosfilt(butter(order, fc, fs=fs, output='sos'), x, axis=0)


def missing_hf(acc, grav, gyr, rng, fs=200.0):
    """Fast (6-25 Hz) content that the replay cannot contain (the markers were low-passed at 6 Hz): soft-tissue
    and strap vibration of a skin-mounted IMU. Band-limited noise scaled by the slow movement; strength drawn per
    trial between pre-filtered (AB06-like) and loose (AB07-like) recordings:
    gyro 6-25 Hz RMS gain 0.02-0.22 x |w|_slow (rad/s), accelerometer 0.04-0.35 x |a|_slow,dyn (g) (AB07: 0.19, 0.32)."""
    from scipy.signal import sosfilt
    n = len(acc)
    bp = butter(4, [6.0, 25.0], btype='band', fs=fs, output='sos')

    def band_noise():
        x = sosfilt(bp, rng.standard_normal((n + 400, 3)), axis=0)[400:]
        return x / (np.sqrt((x ** 2).sum(1).mean()) + 1e-12)          # unit RMS of the vector magnitude
    u = rng.uniform(0.0, 1.0)                                         # one 'attachment looseness' for both sensors
    b_g = 0.02 * (0.22 / 0.02) ** float(np.clip(u + rng.normal(0, 0.1), 0, 1))
    b_a = 0.04 * (0.35 / 0.04) ** float(np.clip(u + rng.normal(0, 0.1), 0, 1))
    w_slow = np.linalg.norm(lowpass(gyr, 6.0, fs), axis=1)
    a_slow = np.linalg.norm(lowpass(acc, 6.0, fs) - grav, axis=1)
    g_hf = band_noise() * (rng.uniform(0.005, 0.03) + b_g * w_slow)[:, None]
    a_hf = band_noise() * (rng.uniform(0.003, 0.012) + b_a * a_slow)[:, None]
    return a_hf, g_hf, dict(looseness=float(u), gyro_gain=float(b_g), acc_gain=float(b_a))


def calibrate_stair_impacts(validation, min_walk_peak=0.05):
    """Stair impact gains relative to walking at 1.1-1.5 m/s, per subject, from the real shank IMU (validation
    results: impact_peak_real_g, contact_speed, contact_label). Subjects whose recordings carry no impacts
    (median walking peak < min_walk_peak g, i.e. pre-filtered) are excluded. Returns per-subject ratios and a
    log-normal fit (mu, sd of log ratio) for ascent and descent."""
    by = {}
    for key, r in validation.items():
        if not r.get('sync_ok'):
            continue
        s = key.split('/')[0]
        d = by.setdefault(s, {1: [], 2: [], 3: []})
        for pk, sp, lab in zip(r.get('impact_peak_real_g', []), r.get('contact_speed', []), r.get('contact_label', [])):
            if lab == 1 and np.isfinite(sp) and 1.1 <= sp <= 1.5:
                d[1].append(pk)
            elif lab in (2, 3):
                d[lab].append(pk)
    per, excluded = {}, []
    for s, d in sorted(by.items()):
        if len(d[1]) < 20 or np.median(d[1]) < min_walk_peak:
            excluded.append(s); continue
        w = float(np.median(d[1]))
        per[s] = dict(walk_peak_g=w, n_walk=len(d[1]),
                      up_ratio=float(np.median(d[2]) / w) if len(d[2]) >= 5 else None, n_up=len(d[2]),
                      down_ratio=float(np.median(d[3]) / w) if len(d[3]) >= 5 else None, n_down=len(d[3]))
    out = dict(subjects=per, excluded=excluded)
    for k in ('up', 'down'):
        v = np.array([p[f'{k}_ratio'] for p in per.values() if p[f'{k}_ratio']])
        if len(v) >= 3:
            out[k] = dict(median=float(np.median(v)), mu=float(np.log(v).mean()), sd=float(max(np.log(v).std(ddof=1), 0.1)),
                          q25=float(np.percentile(v, 25)), q75=float(np.percentile(v, 75)), n=int(len(v)))
    return out


def kabsch(A, B):
    """Rotation R minimising sum ||B_i - R A_i||^2 (rows are vectors)."""
    U, _, Vt = np.linalg.svd(A.T @ B)
    D = np.diag([1.0, 1.0, np.sign(np.linalg.det(Vt.T @ U.T))])
    return Vt.T @ D @ U.T


def lowpass(x, fc, fs=FS_DATA, order=4):
    return sosfiltfilt(butter(order, fc, fs=fs, output='sos'), x, axis=0)


def kinematic_labels(Q, idx, ankle, fs=FS_DATA, mode=None):
    """0 standing, 1 walking, 2 stairs up, 3 stairs down, from kinematics only (the uploaded data have no
    'conditions' labels). Locomotion = knee or ankle-height activity in a 2 s window; stairs = sustained
    vertical pelvis velocity."""
    knee = -np.degrees(Q[:, idx['knee_angle_r']])
    w = int(2 * fs)
    pad = lambda x: np.pad(x, (w // 2, w - w // 2 - 1), mode='edge')
    from numpy.lib.stride_tricks import sliding_window_view as swv
    krng = np.ptp(swv(pad(knee), w), axis=1)
    loco = krng > 25.0
    vy = np.gradient(lowpass(Q[:, idx['pelvis_ty']], 0.7, fs), 1.0 / fs)
    lab = np.where(loco, 1, 0)
    lab[loco & (vy > 0.12)] = 2
    lab[loco & (vy < -0.12)] = 3
    if mode == 'ramp':                      # ramps also raise the pelvis: own classes 4 (up) / 5 (down)
        lab[lab == 2] = 4; lab[lab == 3] = 5
    elif mode is not None and mode != 'stair':
        lab[lab >= 2] = 1                   # treadmill / level ground: vertical drift is not stairs
    return lab


def foot_contacts(ankle, fs=FS_DATA, v_swing=-0.25, v_contact=-0.05):
    """Right-foot contacts: the ankle falls faster than v_swing (m/s) in late swing, then stops (> v_contact)."""
    vy = np.gradient(lowpass(ankle[:, 1], 8.0, fs), 1.0 / fs)
    ev, armed = [], False
    for i in range(1, len(vy)):
        if vy[i] < v_swing:
            armed = True
        elif armed and vy[i] > v_contact:
            ev.append(i)
            armed = False
    return np.array(ev, int)


def walking_speed_at(Q, idx, ankle, ev, fs=FS_DATA):
    """Speed of the body over the stance foot (works on treadmill and overground): |v_pelvis - v_ankle|
    horizontally, averaged over 50-250 ms after each contact."""
    p = np.c_[Q[:, idx['pelvis_tx']], Q[:, idx['pelvis_tz']]]
    a = np.c_[ankle[:, 0], ankle[:, 2]]
    rel = np.gradient(lowpass(p - a, 5.0, fs), 1.0 / fs, axis=0)
    sp = np.linalg.norm(rel, axis=1)
    return np.array([np.median(sp[i + int(0.05 * fs):i + int(0.25 * fs)]) if i + int(0.25 * fs) < len(sp) else np.nan
                     for i in ev])


def align(gyr_sim, acc_sim, gyr_real, acc_real, fc=6.0, w_acc=3.0):
    """Sensor-from-site rotation from gyroscope AND accelerometer (gravity fixes the rotation about the
    dominant sagittal gyro axis, which a gyro-only fit leaves ambiguous). Fit on calibration data only."""
    A = np.r_[lowpass(gyr_sim, fc), w_acc * lowpass(acc_sim, fc)]
    B = np.r_[lowpass(gyr_real, fc), w_acc * lowpass(acc_real, fc)]
    return kabsch(A, B)


def compare(sim, real, fc=6.0):
    """Per-axis Pearson r and amplitude ratio (std sim / std real) after the same zero-phase low-pass."""
    s, r = lowpass(sim, fc), lowpass(real, fc)
    rr = np.array([np.corrcoef(s[:, k], r[:, k])[0, 1] for k in range(3)])
    return rr, s.std(0) / (r.std(0) + 1e-12)


# ----------------------------------------------------------------------------------------------------------------
# dataset labels from the 'conditions' tables (1000 Hz), mapped to KNEIL classes; kinematic labels as fallback
# ----------------------------------------------------------------------------------------------------------------
LABELS = ('standing', 'walking', 'stairs_up', 'stairs_down', 'ramp_up', 'ramp_down')


def map_condition_label(s):
    """Camargo locomotion-mode strings (e.g. 'stand', 'walk', 'turn1', 'stand-walk', 'stairascent',
    'walk-stairdescent', 'rampascent') -> 0..5, or -1 if unknown. Transitions into or out of stairs/ramps count
    as stairs/ramps; turns and stand-walk transitions count as walking."""
    s = str(s).lower()
    if 'stair' in s and 'ascent' in s:
        return 2
    if 'stair' in s and 'descent' in s:
        return 3
    if 'ramp' in s and 'ascent' in s:
        return 4
    if 'ramp' in s and 'descent' in s:
        return 5
    if 'walk' in s or 'turn' in s:
        return 1
    if 'stand' in s or 'idle' in s:
        return 0
    return -1


def conditions_path(ik_path):
    p = ik_path.replace(os.sep + 'ik' + os.sep, os.sep + 'conditions' + os.sep)
    return p if os.path.exists(p) else None


def _flat(x):
    """Unwrap MATLAB cell/char contents (e.g. array(['walk']), [[0.005]], b'walk') to a Python scalar or str."""
    for _ in range(6):
        if isinstance(x, (list, tuple)):
            x = np.asarray(x, dtype=object)
        if not isinstance(x, np.ndarray):
            break
        if x.size == 0:
            return ''
        if x.dtype.kind in 'US' and x.size > 1:          # char array stored letter by letter
            return ''.join(str(c) for c in x.ravel())
        x = x.ravel()[0]
    if isinstance(x, bytes):
        x = x.decode(errors='ignore')
    return x.item() if isinstance(x, np.generic) else x


def conditions_labels(path, header):
    """Labels (0..5, -1 = unknown string) on the time stamps `header` from a conditions .mat file.
    Returns (labels, vocabulary) or (None, reason). Never raises: any problem is returned as the reason."""
    import pandas as pd
    import matio
    try:
        d = matio.load_from_mat(path)
    except Exception as e:
        return None, f'unreadable ({type(e).__name__})'
    reason = 'no label table'
    for name, v in sorted(d.items(), key=lambda kv: (kv[0].lower() != 'labels', kv[0])):   # 'labels' first
        try:
            if not isinstance(v, pd.DataFrame) or len(v) < 2:
                continue
            cols = {c: ([_flat(x) for x in v[c].values] if v[c].dtype == object else list(v[c].values)) for c in v.columns}
            num, txt = [], []
            for c, vals in cols.items():
                (num if pd.to_numeric(pd.Series(vals), errors='coerce').notna().mean() > 0.95 else txt).append(c)
            tcol = 'Header' if 'Header' in num else (num[0] if num else None)
            if tcol is None or not txt:
                continue
            known = {c: np.mean([map_condition_label(x) >= 0 for x in cols[c]]) for c in txt}
            best = max(known, key=known.get)
            if known[best] < 0.5:                         # not a locomotion-label column
                continue
            tt = pd.to_numeric(pd.Series(cols[tcol]), errors='coerce').to_numpy(float)
            ll = np.array([str(x) for x in cols[best]])
            codes = np.array([map_condition_label(x) for x in ll])
            ok = np.isfinite(tt)
            order = np.argsort(tt[ok], kind='stable')
            tt, codes = tt[ok][order], codes[ok][order]
            j = np.clip(np.searchsorted(tt, header, side='right') - 1, 0, len(tt) - 1)
            return codes[j], sorted(set(ll.tolist()))[:40]
        except Exception as e:                            # keep looking; report the last problem
            reason = f'{name}: {type(e).__name__}: {e}'[:120]
    return None, reason


def trial_labels(Q, idx, ankle, ik, ik_path, mode, t_rel):
    """Dataset labels where available (unknown strings filled from kinematics), else kinematic labels.
    Returns (labels, source, agreement_with_kinematics, vocabulary)."""
    kin = kinematic_labels(Q, idx, ankle, fs=1.0 / (t_rel[1] - t_rel[0]), mode=mode)
    cp = conditions_path(ik_path)
    if cp is None:
        return kin, 'kinematics', None, None
    try:
        lab, vocab = conditions_labels(cp, float(ik['Header'].values[0]) + t_rel)
    except Exception as e:                                # labels must never stop a trial
        lab, vocab = None, f'{type(e).__name__}: {e}'[:120]
    if lab is None:
        return kin, f'kinematics ({vocab})', None, None
    ok = lab >= 0
    agree = float(np.mean(lab[ok] == kin[ok])) if ok.any() else None
    return np.where(ok, lab, kin), 'conditions', agree, vocab
''')
print('wrote camargo_replay.py')

In [ ]:
#@title Write `vimp.py` — virtual implant from a shank IMU
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/vimp.py', 'w').write(r'''"""
Virtual KNEIL implant from a thigh IMU and a shank IMU (both with gyroscope).  v1.0, Nov 2026

The implant sits in the tibial insert, i.e. on the tibia, about 2.5 cm below the knee joint centre. A shank IMU is on
the same rigid segment, so the specific force at the implant point P follows from the shank sensor S:
    f_P = f_S + alpha x r + omega x (omega x r)          (sensor frame; r = vector S -> P; gravity cancels)
Steps:
  1. knee hinge axis j1 (thigh frame), j2 (shank frame): minimise sum(|w1 x j1| - |w2 x j2|)^2       (Seel et al.)
  2. knee centre o1, o2 (vectors from each sensor to the joint centre): minimise
     sum(|f1 - G1 o1| - |f2 - G2 o2|)^2 with G(w, a) o = w x (w x o) + a x o, then shift both along the axis to
     the points closest to the sensors                                                            (Seel et al.)
  3. tibia frame in the shank-sensor frame: z = gravity direction while standing still, y = knee axis (sign so that the
     forward leg swing is a rotation about -y, i.e. y points to the subject's left), x = y x z
  4. implant point r = o2 + (medial shift along the axis) - below * z + anterior * x; transfer; rotate to the device
     frame (x anterior, y left, z superior).
Units: acceleration in g, angular velocity in rad/s.
"""
import numpy as np
from scipy.optimize import least_squares
from scipy.signal import butter, sosfiltfilt

G0 = 9.81
VERSION = 'vimp 1.2'


def lowpass(x, fc, fs, order=4):
    return sosfiltfilt(butter(order, fc, fs=fs, output='sos'), x, axis=0)


def unit(v):
    return v / np.maximum(np.linalg.norm(v, axis=-1, keepdims=True), 1e-12)


def sph(t, p):
    return np.array([np.cos(t) * np.cos(p), np.cos(t) * np.sin(p), np.sin(t)])


def gamma(w, a, o):
    """w x (w x o) + a x o for every sample (o fixed, 3,)."""
    return np.cross(w, np.cross(w, o)) + np.cross(a, o)


def moving_mask(w1, w2, fs, thr=1.0):
    """Samples with clear rotation (fitting needs excitation)."""
    m = (np.linalg.norm(w1, axis=1) > thr) | (np.linalg.norm(w2, axis=1) > thr)
    return m


def joint_axis(w1, w2, n_max=3000, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(w1))
    if len(idx) > n_max:
        idx = np.sort(rng.choice(idx, n_max, replace=False))
    W1, W2 = w1[idx], w2[idx]
    def res(x):
        return np.linalg.norm(np.cross(W1, sph(x[0], x[1])), axis=1) - np.linalg.norm(np.cross(W2, sph(x[2], x[3])), axis=1)
    best = None
    for start in rng.uniform(-np.pi, np.pi, (8, 4)):
        r = least_squares(res, start)
        if best is None or r.cost < best.cost:
            best = r
    j1, j2 = sph(best.x[0], best.x[1]), sph(best.x[2], best.x[3])
    if np.corrcoef(W1 @ j1, W2 @ j2)[0, 1] < 0:                  # same rotation direction for both segments
        j1 = -j1
    return j1, j2, float(np.sqrt(best.cost * 2 / len(idx)))


def joint_centre(f1, w1, a1, f2, w2, a2, j1, j2, n_max=3000, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(f1))
    if len(idx) > n_max:
        idx = np.sort(rng.choice(idx, n_max, replace=False))
    F1, W1, A1, F2, W2, A2 = (v[idx] for v in (f1 * G0, w1, a1, f2 * G0, w2, a2))
    def res(x):
        return (np.linalg.norm(F1 - gamma(W1, A1, x[:3]), axis=1) - np.linalg.norm(F2 - gamma(W2, A2, x[3:]), axis=1))
    best = None
    for start in [np.r_[0, 0, 0.2, 0, 0, 0.2], np.r_[0, 0, -0.2, 0, 0, -0.2], np.zeros(6) + 0.05, rng.normal(0, 0.15, 6)]:
        r = least_squares(res, start)
        if best is None or r.cost < best.cost:
            best = r
    o1, o2 = best.x[:3], best.x[3:]
    s = (o1 @ j1 + o2 @ j2) / 2.0                                 # closest to the sensors along the axis
    o1, o2 = o1 - s * j1, o2 - s * j2
    return o1, o2, float(np.sqrt(best.cost * 2 / len(idx)) / G0)


def still_mask(f, w, fs, win=1.0):
    k = max(1, int(win * fs))
    wn = np.convolve(np.linalg.norm(w, axis=1), np.ones(k) / k, mode='same')
    fn = np.convolve(np.abs(np.linalg.norm(f, axis=1) - 1.0), np.ones(k) / k, mode='same')
    return (wn < 0.15) & (fn < 0.05)


def tibia_frame(f2, w2, j2, fs):
    """Rows = tibia x (anterior), y (left), z (superior) in shank-sensor coordinates."""
    st = still_mask(f2, w2, fs)
    g = f2[st].mean(0) if st.sum() > fs else lowpass(f2, 0.3, fs).mean(0)
    z = unit(g)
    y = unit(j2 - (j2 @ z) * z)
    wy = w2 @ y                                                   # forward swing = big rotation about -y
    peak = np.argmax(np.abs(wy))
    hi = np.abs(wy) > 0.6 * np.abs(wy[peak])
    if np.mean(wy[hi]) > 0:
        y = -y
    x = np.cross(y, z)
    return np.stack([x, y, z]), int(st.sum())


def transfer(f, w, a, r):
    """Specific force [g] at the point r (sensor frame, m) of the same rigid segment."""
    return f + gamma(w, a, r) / G0


def calibrate_up(f, w, fs):
    """Gravity direction (sensor coordinates) during quiet standing, or None if there is no still period of 1 s.
    Use one calibration per person and sensor for all their trials (the sensor stays on the leg)."""
    st = still_mask(lowpass(f, 15, fs), lowpass(w, 15, fs), fs)
    if st.sum() < fs:
        return None
    return unit(lowpass(f, 15, fs)[st].mean(0))


def tibia_frame_pca(w, fs, z_up, fc=15.0):
    """Rows x (anterior), y (left), z (superior) in sensor coordinates. y = main rotation axis of the shank while
    moving (sagittal rotation dominates), sign so that the big forward-swing peaks rotate about -y."""
    wf = lowpass(w, fc, fs)
    mv = np.linalg.norm(wf, axis=1) > 1.0
    if mv.sum() < fs:
        mv = np.linalg.norm(wf, axis=1) > 0.3
    pc = np.linalg.eigh(np.cov(wf[mv].T))[1][:, -1]
    z = unit(z_up)
    y = unit(pc - (pc @ z) * z)
    wy = wf @ y
    hi = np.abs(wy) > 0.6 * np.abs(wy).max()
    if wy[hi].mean() > 0:
        y = -y
    return np.stack([np.cross(y, z), y, z])


def virtual_implant(f, w, fs, z_up=None, distance=0.22, anterior=-0.02, medial=0.0, side='R', fc=15.0, T=None):
    """Implant specific force [g] in the device frame (x anterior, y left, z superior) from ONE shank IMU.
    z_up: standing gravity direction of this sensor (calibrate_up); if None, quiet periods of this recording are used,
    and the trial-mean direction as the last resort (biased by the forward lean of the shank while walking).
    distance: sensor-to-implant distance along the tibia axis (implant above the sensor); anterior/medial offsets of
    the implant relative to the sensor (m). Validated on Camargo (real shank IMU vs implant from the replay):
    the knee-centre fit of Seel et al. gave a wrong lever arm on these data, so it is not used."""
    how = 'calibrated'
    if z_up is None:
        z_up = calibrate_up(f, w, fs); how = 'still periods of this recording'
    if z_up is None:
        z_up = unit(lowpass(f, 0.3, fs).mean(0)); how = 'trial mean (biased by the walking lean)'
    if T is None:
        T = tibia_frame_pca(w, fs, z_up, fc)
    x, y, z = T
    med = medial if side.upper().startswith('R') else -medial          # medial = +y on the right leg, -y on the left
    r = distance * z + anterior * x + med * y
    wf = lowpass(w, fc, fs)
    fP = transfer(f, wf, np.gradient(wf, 1 / fs, axis=0), r)
    return fP @ T.T, dict(T=T, r=r, up_source=how)


def person_frame(standing, walking, fs):
    """One tibia frame per person and sensor (the sensor does not move between their tasks).
    standing: list of (f, w) recordings that contain quiet standing; walking: list of (f, w) walking recordings.
    z = standing gravity direction; y = main rotation axis while walking, signed by the forward swing; x = y x z.
    Returns (T or None, note). Squats, sit-to-stand or seated tasks then use this frame: their own motion does not
    say which way is forward (stage 1b found x flipped in squats with a per-recording frame)."""
    z = None
    for f, w in standing + walking:
        z = calibrate_up(f, w, fs)
        if z is not None:
            break
    if z is None:
        return None, 'no quiet standing found'
    if not walking:
        return None, 'no walking recording to set the forward direction'
    W = np.vstack([w for f, w in walking])
    return tibia_frame_pca(W, fs, z), f'standing from {len(standing)} rec., forward from {len(walking)} walking rec.'
''')
print('wrote vimp.py')

In [ ]:
#@title Write `stage01.py` — stages 0 and 1
import os
os.makedirs('/content/kneil', exist_ok=True)
open('/content/kneil/stage01.py', 'w').write(r'''"""
KNEIL virtual implant, stages 0 and 1 (Colab).  v1.0, Nov 2026
Stage 0: inventory of every dataset folder (found? format as expected? sensors, rates, units).
Stage 1: validation of the virtual implant (vimp.py):
   1a  Camargo: virtual implant from the REAL shank IMU vs the implant signal of the replay of the measured joint
       angles (same point, same frame). Variants: calibrated (per person), no calibration, no transfer.
   1b  Two sensors on the same tibia must give the same implant signal:
       KneE-PAD (tibialis anterior vs gastrocnemius sensor) and WearGait-PD (lateral shank vs ankle sensor).
All results go to <DATA_ROOT>/KNEIL_results/virtual_implant/.
"""
import glob, io, json, os, re, shutil, sys, time, traceback, zipfile
import numpy as np
import pandas as pd

import vimp as V

VERSION = 'stage01 1.4'
WG_NAME = re.compile(r'([A-Z]*(?:HC|PD)\d+)_(.+)\.csv$', re.I)   # full person id (e.g. HC100, WHC001, PD012) and task
wg_is_control = lambda pid: 'HC' in pid.upper()
TILT_OK = (-5.0, 35.0)                     # plausible mean forward lean of the tibia while moving (deg)
LP = 6.0                                   # the device's input low-pass (Hz); all comparisons after it


LOCAL_CACHE = '/content/kneil_cache'


def remount():
    """Re-mount Google Drive after 'Transport endpoint is not connected' (Colab only)."""
    try:
        from google.colab import drive
        print('  (Drive connection lost: remounting)', flush=True)
        drive.mount('/content/drive', force_remount=True)
        time.sleep(3)
        return True
    except Exception as e:
        print(f'  (remount failed: {e!r})', flush=True)
        return False


def with_retry(fn, what, tries=4):
    for k in range(tries):
        try:
            return fn()
        except OSError as e:
            if k == tries - 1:
                raise
            print(f'  {what}: {e!r} -> retry {k + 1}', flush=True)
            if getattr(e, 'errno', None) in (107, 5, 103, 110) or 'endpoint' in str(e):
                remount()
            time.sleep(5 * (k + 1))


def local_copy(path, sub=''):
    """Copy a Drive file to the local disk once (with retries); returns the local path."""
    d = os.path.join(LOCAL_CACHE, sub); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.basename(path))
    if os.path.exists(dst):
        return dst
    def go():
        shutil.copyfile(path, dst + '.part'); os.replace(dst + '.part', dst)
    with_retry(go, f'copy {os.path.basename(path)}')
    return dst


def save_csv(df, path):
    with_retry(lambda: df.to_csv(path, index=False), f'save {os.path.basename(path)}')


class Tee:
    """Write to the notebook output and to a log file. Use: with Tee(path): ..."""
    """The log is written on the local disk and copied to Drive at the end (a Drive drop cannot crash it)."""
    def __init__(self, path):
        self.path = path; self.f = None; self.out = None
        os.makedirs(LOCAL_CACHE, exist_ok=True)
        self.local = os.path.join(LOCAL_CACHE, os.path.basename(path))
    def __enter__(self):
        self.out = sys.stdout
        if not os.path.exists(self.local) and os.path.exists(self.path):
            try:
                shutil.copyfile(self.path, self.local)
            except OSError:
                pass
        self.f = open(self.local, 'a', encoding='utf-8'); sys.stdout = self
        return self
    def __exit__(self, *exc):
        if exc[0] is not None:
            self.write(''.join(traceback.format_exception(*exc)))
        sys.stdout = self.out; self.f.close()
        try:
            with_retry(lambda: shutil.copyfile(self.local, self.path), 'copy the log to Drive')
        except OSError as e:
            print(f'log kept locally at {self.local} ({e!r})')
        return False
    def write(self, s):
        self.out.write(s); self.f.write(s); self.f.flush()
    def flush(self):
        self.out.flush(); self.f.flush()


def find_dir(root, *patterns):
    for p in patterns:
        hits = sorted(glob.glob(os.path.join(root, p)))
        if hits:
            return hits[0]
    return None


def walk_files(top, ext, limit=200000):
    out = []
    for dp, dn, fn in os.walk(top):
        out += [os.path.join(dp, f) for f in fn if f.lower().endswith(ext) and not f.startswith('~$')]
        if len(out) > limit:
            break
    return sorted(out)


def parse_time(col):
    """Numbers, or text such as '0 sec', '0.01 sec', '10 ms', '00:00:01.250'. Returns seconds and a note."""
    if pd.api.types.is_numeric_dtype(pd.Series(col)):
        return np.asarray(col, float), 'numeric'
    s = pd.Series(col).astype(str).str.strip()
    if s.str.contains(r'^\d+:\d+:\d+', regex=True).mean() > 0.5:
        return pd.to_timedelta(s, errors='coerce').dt.total_seconds().to_numpy(), 'hh:mm:ss'
    num = pd.to_numeric(s.str.extract(r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')[0], errors='coerce').to_numpy(float)
    unit = s.str.extract(r'([a-zA-Z]+)\s*$')[0].dropna()
    u = unit.str.lower().mode().iloc[0] if len(unit) else ''
    if u in ('ms', 'msec', 'millisecond', 'milliseconds'):
        return num / 1000.0, f"text '{s.iloc[0]}' (ms)"
    return num, f"text '{s.iloc[0]}' ({u or 'no unit'}, read as s)"


def fs_from_time(t):
    t = np.asarray(t, float)
    dt = float(np.median(np.diff(t[np.isfinite(t)][:5000])))
    if dt <= 0:
        return None, 'time not increasing'
    if dt < 1.0:
        return 1.0 / dt, 's'
    return 1000.0 / dt, 'ms'


def to_g(acc):
    med = float(np.nanmedian(np.linalg.norm(acc, axis=1)))
    if 6.0 < med < 13.0:
        return acc / 9.81, 'm/s^2'
    return acc, 'g'


def to_rad(gyr):
    p99 = float(np.nanpercentile(np.linalg.norm(gyr, axis=1), 99))
    if p99 > 25.0:
        return np.radians(gyr), 'deg/s'
    return gyr, 'rad/s'


def fill(x):
    x = np.asarray(x, float)
    if not np.isnan(x).any():
        return x
    idx = np.arange(len(x))
    for j in range(x.shape[1]):
        ok = np.isfinite(x[:, j])
        x[:, j] = np.interp(idx, idx[ok], x[ok, j]) if ok.sum() > 1 else 0.0
    return x


# ---------------------------------------------------------------- readers
WG_SENSORS = ('R_LatShank', 'L_LatShank', 'R_Ankle', 'L_Ankle', 'R_MidLatThigh', 'L_MidLatThigh', 'R_Wrist', 'L_Wrist', 'LowerBack')


def wg_read(path, sensors=WG_SENSORS, nrows=None, cache=True):
    if cache and path.startswith('/content/drive'):
        path = local_copy(path, 'weargait')
    cols = pd.read_csv(path, nrows=0).columns
    tcol = next((c for c in cols if c.strip().lower() == 'time'), None)
    use = ([tcol] if tcol else []) + [c for c in cols if any(c.startswith(s + '_Acc_') or c.startswith(s + '_Gyr_') for s in sensors)]
    df = pd.read_csv(path, usecols=use, nrows=nrows)
    if tcol:
        tsec, tnote = parse_time(df[tcol].to_numpy())
        fs, tunit = fs_from_time(tsec)
        tunit = f'{tnote} -> {tunit}'
    else:
        fs, tunit = None, 'no time column'
    if not fs or not np.isfinite(fs):
        raise ValueError(f'cannot get the sampling rate from the time column ({tunit})')
    out, units = {}, {}
    for s in sensors:
        a = [f'{s}_Acc_{k}' for k in 'XYZ']; g = [f'{s}_Gyr_{k}' for k in 'XYZ']
        if all(c in df for c in a + g):
            acc, ua = to_g(fill(df[a].to_numpy(float))); gyr, ug = to_rad(fill(df[g].to_numpy(float)))
            out[s] = dict(acc=acc, gyr=gyr); units[s] = (ua, ug)
    return out, fs, dict(time_unit=tunit, units=units, n=len(df))


KP_FS = 148.148
KP_UNITS = {'R_thigh_front': 0, 'R_thigh_back': 1, 'R_shank_TA': 2, 'R_shank_GA': 3,
            'L_thigh_front': 4, 'L_thigh_back': 5, 'L_shank_TA': 6, 'L_shank_GA': 7}
KP_TASKS = {0: 'squat', 1: 'squat (weight on healthy leg)', 2: 'squat (injured leg in front)', 3: 'seated extension',
            4: 'seated extension (partial)', 5: 'seated extension (limb lifted)', 6: 'walking', 7: 'walking (knee not extended)',
            8: 'walking (extension + hip abduction)'}


def kp_members(zpath):
    with zipfile.ZipFile(zpath) as z:
        names = [n for n in z.namelist() if n.replace('\\', '/').endswith('imu.npy')]
    rows = []
    for n in names:
        m = re.search(r'Subject_(\d+)[/\\](\d+)[/\\]Trial_(\d+)', n)
        if m:
            rows.append(dict(member=n, subject=int(m.group(1)), label=int(m.group(2)), trial=int(m.group(3))))
    return pd.DataFrame(rows)


def kp_load(z, member):
    x = np.load(io.BytesIO(z.read(member)))
    if x.shape[0] != 48 and x.shape[1] == 48:
        x = x.T
    out = {}
    for name, u in KP_UNITS.items():
        acc, ua = to_g(x[6 * u:6 * u + 3].T.astype(float)); gyr, ug = to_rad(x[6 * u + 3:6 * u + 6].T.astype(float))
        out[name] = dict(acc=fill(acc), gyr=fill(gyr), units=(ua, ug))
    return out


# ---------------------------------------------------------------- stage 0
def inventory(root, out_dir):
    res = {}
    def rec(name, ok, **facts):
        res[name] = dict(status='OK' if ok else 'PROBLEM', **facts)
        print(f"{name:22s} {'OK     ' if ok else 'PROBLEM'} " + '; '.join(f'{k}: {v}' for k, v in facts.items()), flush=True)
    def guard(name, fn):
        try:
            fn()
        except Exception as e:
            rec(name, False, error=repr(e)[:300])
    def camargo():
        d = find_dir(root, 'Camargo*/Data', 'Camargo*')
        zips = sorted(f for f in os.listdir(d) if re.fullmatch(r'ab\d+\.zip', f, re.I)) if d else []
        rec('Camargo', len(zips) > 0, folder=d, subject_zips=len(zips))
    def weargait():
        d = find_dir(root, 'WearGait*')
        man = os.path.join(d, 'manifest.csv') if d else None
        files = walk_files(d, '.csv') if d else []
        data = [f for f in files if WG_NAME.search(os.path.basename(f))]
        tasks = sorted({WG_NAME.search(os.path.basename(f)).group(2) for f in data})
        people = sorted({WG_NAME.search(os.path.basename(f)).group(1).upper() for f in data})
        print('    WearGait-PD example file names:', [os.path.basename(f) for f in data[:3]] + [os.path.basename(f) for f in data if not re.match(r'(HC|PD)\d+_', os.path.basename(f), re.I)][:3], flush=True)
        sp = next((f for f in data if f.endswith('_SelfPace.csv')), None)
        S, fs, info = wg_read(sp, nrows=4000) if sp else ({}, None, {})
        rec('WearGait-PD', bool(data) and bool(S), folder=d, csv_files=len(data), people=len(people),
            controls=sum(wg_is_control(p) for p in people), parkinson=sum(not wg_is_control(p) for p in people),
            id_prefixes=sorted({re.match(r'[A-Z]*', p).group(0) for p in people}), tasks=tasks,
            sensors_found=sorted(S), fs_hz=round(fs, 2) if fs else None, time_unit=info.get('time_unit'),
            units={k: v for k, v in info.get('units', {}).items()}, manifest=bool(man and os.path.exists(man)))
    def kneepad():
        zp = None
        for d in (find_dir(root, 'Clement*'), find_dir(root, 'KneE*', 'ROAM*')):
            if d:
                hits = walk_files(d, '.zip')
                zp = hits[0] if hits else zp
        df = kp_members(zp) if zp else pd.DataFrame()
        shape = None
        if len(df):
            with zipfile.ZipFile(zp) as z:
                shape = list(np.load(io.BytesIO(z.read(df.member.iloc[0]))).shape)
        rec('KneE-PAD', len(df) > 0 and shape is not None and 48 in shape, zip=zp, trials=len(df),
            subjects=int(df.subject.nunique()) if len(df) else 0, labels=sorted(df.label.unique().tolist()) if len(df) else [],
            imu_shape=shape, fs_hz=KP_FS)
    def marpino():
        d = find_dir(root, 'Marpino*')
        x = walk_files(d, '.xlsx') if d else []
        rec('Marpino', len(x) > 0, folder=d, workbooks=len(x))
    def ceti():
        d = find_dir(root, 'CeTI*')
        zp = (walk_files(d, '.zip') or [None])[0] if d else None
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            mot = [n for n in names if n.endswith('_motion.tsv')]; ch = [n for n in names if n.endswith('_channels.tsv')]
            js = [n for n in names if n.endswith('_motion.json')]
            chan = pd.read_csv(io.BytesIO(z.read(ch[0])), sep='\t') if ch else pd.DataFrame()
            meta = json.loads(z.read(js[0])) if js else {}
        types = chan['type'].value_counts().to_dict() if 'type' in chan else {}
        rec('CeTI-Age-Kinematics', bool(mot), zip=zp, motion_files=len(mot), channels_in_first=len(chan),
            channel_types=types, first_channels=chan['name'].head(12).tolist() if 'name' in chan else [],
            fs_hz=meta.get('SamplingFrequency'), tracking_systems=meta.get('TrackingSystemName'))
    def pamap2():
        d = find_dir(root, 'PAMAP2*')
        zp = (walk_files(d, '.zip') or [None])[0] if d else None
        with zipfile.ZipFile(zp) as z:
            dat = [n for n in z.namelist() if re.search(r'Protocol/subject\d+\.dat$', n)]
            first = z.read(dat[0])[:400000].decode(errors='ignore').splitlines()[:2000] if dat else []
        ncol = int(np.median([len(l.split()) for l in first])) if first else 0
        rec('PAMAP2', bool(dat) and ncol == 54, zip=zp, protocol_files=len(dat), columns=ncol, expected_columns=54)
    def mhealth():
        d = find_dir(root, 'MHEALTH*')
        logs = [f for f in walk_files(d, '.log') if 'subject' in f.lower()] if d else []
        x = pd.read_csv(logs[0], sep=r'\s+', header=None, nrows=2000) if logs else pd.DataFrame()
        rec('MHEALTH', bool(logs) and x.shape[1] == 24, folder=d, subject_logs=len(logs), columns=x.shape[1], expected_columns=24)
    def har():
        for name, pat in (('HAR70+', 'har70*'), ('HARTH', 'harth*')):
            d = find_dir(root, pat)
            c = walk_files(d, '.csv') if d else []
            rec(name, len(c) > 0, folder=d, csv_files=len(c))
    def caicedo():
        d = find_dir(root, 'Rodr*')
        c = [f for f in walk_files(d, '.csv') if re.search(r'Subject\d+_\d+', os.path.basename(f))] if d else []
        rec('Caicedo/Rodriguez', len(c) > 0, folder=d, walk_csvs=len(c))
    for name, fn in (('Camargo', camargo), ('WearGait-PD', weargait), ('KneE-PAD', kneepad), ('Marpino', marpino),
                     ('CeTI-Age-Kinematics', ceti), ('PAMAP2', pamap2), ('MHEALTH', mhealth), ('HAR', har), ('Caicedo', caicedo)):
        guard(name, fn)
    json.dump(res, open(os.path.join(out_dir, 'stage0_inventory.json'), 'w'), indent=1, default=str)
    return res


# ---------------------------------------------------------------- stage 1a: Camargo
def _lag(a, b, max_lag):
    from scipy.signal import correlate
    n = min(len(a), len(b)); a = a[:n] - a[:n].mean(); b = b[:n] - b[:n].mean()
    c = correlate(a, b, mode='full', method='fft')
    seg = c[n - 1 - max_lag:n + max_lag]
    return int(np.argmax(seg) - max_lag)


def moving_tilt(p, fs):
    """Mean sagittal tilt (deg, + = top of the tibia forward) of an implant signal over its moving samples."""
    a = V.lowpass(p, LP, fs)
    mv = np.convolve(np.linalg.norm(a - a.mean(0), axis=1), np.ones(int(fs)) / fs, mode='same') > 0.1
    a = a[mv] if mv.sum() > fs else a
    return float(np.degrees(np.arctan2(-a[:, 0].mean(), a[:, 2].mean())))


def plausible(p, fs):
    t = moving_tilt(p, fs)
    return dict(implant_tilt_deg=round(t, 1), plausible=bool(TILT_OK[0] <= t <= TILT_OK[1]))


def metrics(p, q, fs):
    a, b = V.lowpass(p, LP, fs), V.lowpass(q, LP, fs)
    r = [float(np.corrcoef(a[:, i], b[:, i])[0, 1]) for i in range(3)]
    ra = float(np.corrcoef(np.linalg.norm(a, axis=1), np.linalg.norm(b, axis=1))[0, 1])
    tilt = lambda u: np.degrees(np.arctan2(-u[:, 0].mean(), u[:, 2].mean()))
    return dict(rms_g=float(np.sqrt(((a - b) ** 2).sum(1).mean())), r_x=r[0], r_y=r[1], r_z=r[2], r_mag=ra,
                tilt_diff_deg=float(tilt(a) - tilt(b)))


def camargo_subject(subj, zip_path, local, modes, per_mode, log=print):
    import camargo_replay as C
    import mujoco
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    _extract(zip_path, subj, local, modes)
    m, info = C.model_for(subj)
    fs = C.FS_DATA
    imu_site, shin_site = m.site('imu').id, m.site('shin_r').id
    dpos = m.site_pos[imu_site] - m.site_pos[shin_site]              # tibia body frame: x anterior, y superior
    distance, anterior = float(dpos[1]), float(dpos[0])
    cols = lambda seg, kind: [f'{seg}_{kind}_{a}' for a in 'XYZ']
    z_up, cal_from = None, None
    for name, pik, pimu in C.find_trials(local, subj, 'levelground') + C.find_trials(local, subj, 'treadmill'):
        imu = C.load_table(pimu)
        z_up = V.calibrate_up(imu[cols('shank', 'Accel')].values, imu[cols('shank', 'Gyro')].values, fs)
        if z_up is not None:
            cal_from = name; break
    rows = []
    for mode in modes:
        trials = C.find_trials(local, subj, mode)
        for name, pik, pimu in trials[::max(1, len(trials) // per_mode)][:per_mode]:
            try:
                ik, imu = C.load_table(pik), C.load_table(pimu)
                t, Q = C.ik_to_Q(m, info, ik)
                o = C.replay(m, Q, fs, sensors=('acc_imu', 'gyr_shin_r'))
                n = min(len(t), len(imu))
                f, w = imu[cols('shank', 'Accel')].values[:n], imu[cols('shank', 'Gyro')].values[:n]
                tgt, ws = o['acc_imu'][:n], o['gyr_shin_r'][:n]
                L = _lag(np.linalg.norm(V.lowpass(w, LP, fs), axis=1), np.linalg.norm(V.lowpass(ws, LP, fs), axis=1), int(fs))
                sr, ss = (slice(L, None), slice(None, n - L)) if L >= 0 else (slice(None, n + L), slice(-L, None))
                f, w, tgt = f[sr], w[sr], tgt[ss]
                for variant, kw in (('calibrated', dict(z_up=z_up)), ('no calibration', dict(z_up=None)), ('no transfer', dict(z_up=z_up, distance=0.0, anterior=0.0))):
                    if variant == 'calibrated' and z_up is None:
                        continue
                    kw.setdefault('distance', distance); kw.setdefault('anterior', anterior)
                    p, g = V.virtual_implant(f, w, fs, **kw)
                    rows.append(dict(subject=subj, mode=mode, trial=name, variant=variant, lag=L, up_source=g['up_source'],
                                     seconds=round(len(f) / fs, 1), **plausible(p, fs), **metrics(p, tgt, fs)))
                r = [x for x in rows if x['trial'] == name and x['variant'] == ('calibrated' if z_up is not None else 'no calibration')][-1]
                log(f"  {subj} {name:32s} |a| r {r['r_mag']:.2f}, r x/z {r['r_x']:+.2f}/{r['r_z']:+.2f}, RMS {r['rms_g']:.3f} g, "
                    f"tilt {r['implant_tilt_deg']:+.0f} deg{'' if r['plausible'] else '  <- FLAGGED (implausible lean: sensor data suspect)'}")
            except Exception as e:
                log(f'  {subj} {name}: FAILED {e!r}')
    return rows, dict(subject=subj, calibration_trial=cal_from, distance_m=distance, anterior_m=anterior)


def _extract(zip_path, subj, local, modes, sensors=('ik', 'imu')):
    dst_root = os.path.join(local, subj); flag = os.path.join(dst_root, '.extracted.json')
    if os.path.exists(flag):
        return
    n = 0
    with zipfile.ZipFile(zip_path) as z:
        for info in z.infolist():
            parts = info.filename.replace('\\', '/').split('/')
            if info.is_dir() or len(parts) < 3 or not parts[-1].lower().endswith('.mat'):
                continue
            mode, sensor = parts[-3].lower(), parts[-2].lower()
            if mode in modes and sensor in sensors:
                dst = os.path.join(dst_root, mode, sensor, parts[-1]); os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(info) as src, open(dst + '.part', 'wb') as fo:
                    shutil.copyfileobj(src, fo, 1 << 20)
                os.replace(dst + '.part', dst); n += 1
    if not n:
        raise RuntimeError(f'{subj}: no <mode>/<ik|imu>/<trial>.mat files in {zip_path}')
    os.makedirs(dst_root, exist_ok=True); json.dump(dict(files=n), open(flag, 'w'))


def stage1a(root, out_dir, subjects, per_mode=4, modes=('treadmill', 'levelground', 'stair'), local='/content/camargo_local', keep_local=False):
    d = find_dir(root, 'Camargo*/Data', 'Camargo*')
    zips = {re.match(r'(ab\d+)', f, re.I).group(1).lower(): os.path.join(d, f) for f in os.listdir(d) if re.fullmatch(r'ab\d+\.zip', f, re.I)}
    subs = sorted(zips) if subjects == 'all' else [s for s in subjects if s in zips]
    csv = os.path.join(out_dir, 'stage1a_camargo_trials.csv')
    done = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    geo = []
    for s in subs:
        if len(done) and (done.subject == s).any():
            print(f'{s}: already done (delete stage1a_camargo_trials.csv to redo)'); continue
        t0 = time.time()
        try:
            rows, g = camargo_subject(s, zips[s], local, modes, per_mode)
            geo.append(g)
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(csv, index=False)
            print(f'{s}: {len({r["trial"] for r in rows})} trials in {time.time() - t0:.0f} s (calibrated from {g["calibration_trial"]})', flush=True)
        except Exception as e:
            print(f'{s}: FAILED {e!r}\n{traceback.format_exc()}')
        if not keep_local:
            shutil.rmtree(os.path.join(local, s), ignore_errors=True)
    if geo:
        json.dump(geo, open(os.path.join(out_dir, 'stage1a_geometry.json'), 'w'), indent=1)
    return done


# ---------------------------------------------------------------- stage 1b: two sensors on one tibia
KP_GEOM = {'TA': dict(distance=0.12, anterior=-0.03, medial=0.03), 'GA': dict(distance=0.14, anterior=0.07, medial=0.0)}
WG_GEOM = {'LatShank': dict(distance=0.18, anterior=0.0, medial=0.04), 'Ankle': dict(distance=0.33, anterior=0.0, medial=0.04)}


def stage1b_kneepad(root, out_dir, max_subjects=10, labels=(0, 6), log=print):
    zp = None
    d = find_dir(root, 'Clement*')
    if d:
        zp = (walk_files(d, '.zip') or [None])[0]
    if not zp:
        log('KneE-PAD zip not found'); return []
    csv = os.path.join(out_dir, 'stage1b_kneepad.csv')
    old = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    done_subj = set(old.subject.unique()) if len(old) else set()
    log(f'  copying the KneE-PAD zip to the local disk ({os.path.getsize(zp) / 1e9:.2f} GB, once)'); zp = local_copy(zp, 'kneepad')
    df = kp_members(zp)
    rows = old.to_dict(orient='records') if len(old) else []
    with zipfile.ZipFile(zp) as z:
        for s in sorted(df.subject.unique())[:max_subjects]:
            if s in done_subj:
                log(f'  KneE-PAD subject {s}: already done'); continue
            sd = df[df.subject == s]
            loaded = {r.member: kp_load(z, r.member) for r in sd[sd.label.isin(list(range(9)))].itertuples()}
            for side in 'RL':
                cal = {}
                for kind in ('TA', 'GA'):
                    nm = f'{side}_shank_{kind}'
                    f = np.vstack([v[nm]['acc'] for v in loaded.values()]); w = np.vstack([v[nm]['gyr'] for v in loaded.values()])
                    cal[kind] = V.calibrate_up(f, w, KP_FS)
                for r in sd[sd.label.isin(labels)].itertuples():
                    v = loaded[r.member]
                    try:
                        imp = {}
                        for kind in ('TA', 'GA'):
                            nm = f'{side}_shank_{kind}'
                            imp[kind], g = V.virtual_implant(v[nm]['acc'], v[nm]['gyr'], KP_FS, z_up=cal[kind], side=side, **KP_GEOM[kind])
                        pa, pb = plausible(imp['TA'], KP_FS), plausible(imp['GA'], KP_FS)
                        rows.append(dict(dataset='KneE-PAD', subject=s, side=side, task=KP_TASKS.get(r.label, r.label), trial=r.trial,
                                         pair='TA vs GA', calibrated=all(c is not None for c in cal.values()),
                                         tilt_a=pa['implant_tilt_deg'], tilt_b=pb['implant_tilt_deg'], plausible=pa['plausible'] and pb['plausible'],
                                         **metrics(imp['TA'], imp['GA'], KP_FS)))
                    except Exception as e:
                        log(f'  KneE-PAD S{s} {side} {r.member}: {e!r}')
            log(f'  KneE-PAD subject {s}: {sum(1 for x in rows if x["subject"] == s)} comparisons')
            save_csv(pd.DataFrame(rows), csv)
    return rows


def stage1b_weargait(root, out_dir, max_people=10, log=print):
    d = find_dir(root, 'WearGait*')
    files = [f for f in walk_files(d, '.csv') if WG_NAME.search(os.path.basename(f))] if d else []
    by = {}
    for f in files:
        p = WG_NAME.search(os.path.basename(f))
        by.setdefault(p.group(1).upper(), {})[p.group(2)] = f
    csv = os.path.join(out_dir, 'stage1b_weargait.csv')
    old = pd.read_csv(csv) if os.path.exists(csv) else pd.DataFrame()
    rows = old.to_dict(orient='records') if len(old) else []
    done_p = set(old.subject.astype(str)) if len(old) else set()
    people = sorted(by)
    pick = [p for p in people if wg_is_control(p)][:max_people // 2] + [p for p in people if not wg_is_control(p)][:max_people - max_people // 2]
    for p in pick:
        t = by[p]
        if 'SelfPace' not in t or p in done_p:
            if p in done_p:
                log(f'  WearGait-PD {p}: already done')
            continue
        try:
            sens = tuple(f'{s}_{k}' for s in 'RL' for k in ('LatShank', 'Ankle'))
            cal = {}
            if 'Balance' in t:
                B, fsb, _ = wg_read(t['Balance'], sensors=sens)
                cal = {k: V.calibrate_up(v['acc'], v['gyr'], fsb) for k, v in B.items()}
            for task in ('SelfPace', 'HurriedPace', 'TUG'):
                if task not in t:
                    continue
                S, fs, info = wg_read(t[task], sensors=sens)
                for side in 'RL':
                    a, b = f'{side}_LatShank', f'{side}_Ankle'
                    if a not in S or b not in S:
                        continue
                    pa, _ = V.virtual_implant(S[a]['acc'], S[a]['gyr'], fs, z_up=cal.get(a), side=side, **WG_GEOM['LatShank'])
                    pb, _ = V.virtual_implant(S[b]['acc'], S[b]['gyr'], fs, z_up=cal.get(b), side=side, **WG_GEOM['Ankle'])
                    qa, qb = plausible(pa, fs), plausible(pb, fs)
                    rows.append(dict(dataset='WearGait-PD', subject=p, side=side, task=task, trial=1, pair='LatShank vs Ankle',
                                     calibrated=cal.get(a) is not None and cal.get(b) is not None, tilt_a=qa['implant_tilt_deg'],
                                     tilt_b=qb['implant_tilt_deg'], plausible=qa['plausible'] and qb['plausible'], **metrics(pa, pb, fs)))
            log(f'  WearGait-PD {p}: {sum(1 for x in rows if x["subject"] == p)} comparisons')
            save_csv(pd.DataFrame(rows), csv)
        except Exception as e:
            log(f'  WearGait-PD {p}: FAILED {e!r}')
        finally:
            shutil.rmtree(os.path.join(LOCAL_CACHE, 'weargait'), ignore_errors=True)
    return rows


def stage1b_all(root, out_dir, kneepad_subjects=10, weargait_people=10):
    a = stage1b_kneepad(root, out_dir, max_subjects=kneepad_subjects)
    b = stage1b_weargait(root, out_dir, max_people=weargait_people)
    print(f'stage 1b: {len(a)} KneE-PAD and {len(b)} WearGait-PD comparisons saved')
    return a + b


# ---------------------------------------------------------------- summary
def summarise(out_dir):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    a = pd.read_csv(os.path.join(out_dir, 'stage1a_camargo_trials.csv')) if os.path.exists(os.path.join(out_dir, 'stage1a_camargo_trials.csv')) else pd.DataFrame()
    parts = [pd.read_csv(os.path.join(out_dir, f)) for f in ('stage1b_kneepad.csv', 'stage1b_weargait.csv') if os.path.exists(os.path.join(out_dir, f))]
    b = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    summ = {}
    print('\n=== STAGE 1 SUMMARY (all after the 6 Hz low-pass; medians) ===')
    if len(a):
        bad = a[(a.variant == 'calibrated') & ~a.plausible.astype(bool)]
        print(f'\n1a  Camargo: virtual implant from the real shank IMU vs implant of the replayed joint angles')
        print(f'    flagged as implausible (left out below): {len(bad)} of {(a.variant == "calibrated").sum()} trials'
              + (f' -> {sorted(set(bad.subject + " " + bad["mode"]))}' if len(bad) else ''))
        flagged = set(bad.trial + bad.subject)
        a = a[~(a.trial + a.subject).isin(flagged)]
        t = a.groupby('variant')[['r_mag', 'r_x', 'r_y', 'r_z', 'rms_g', 'tilt_diff_deg']].median().round(3)
        t['trials'] = a.groupby('variant').size(); print(t.to_string())
        print('\n    calibrated, by mode:'); print(a[a.variant == 'calibrated'].groupby('mode')[['r_mag', 'r_x', 'r_z', 'rms_g']].median().round(3).to_string())
        summ['camargo'] = t.reset_index().to_dict(orient='records')
    if len(b):
        nb = int((~b.plausible.astype(bool)).sum()) if 'plausible' in b else 0
        print(f'\n1b  Two sensors on the same tibia (same implant point computed from each); flagged and left out: {nb} of {len(b)}')
        b = b[b.plausible.astype(bool)] if 'plausible' in b else b
        t2 = b.groupby(['dataset', 'pair', 'task'])[['r_mag', 'r_x', 'r_z', 'rms_g', 'tilt_diff_deg']].median().round(3)
        t2['n'] = b.groupby(['dataset', 'pair', 'task']).size(); print(t2.to_string())
        summ['two_sensor'] = t2.reset_index().to_dict(orient='records')
    json.dump(summ, open(os.path.join(out_dir, 'stage1_summary.json'), 'w'), indent=1, default=float)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    if len(a):
        vs = ['calibrated', 'no calibration', 'no transfer']
        data = [a[a.variant == v]['r_mag'].dropna() for v in vs]
        ax[0].boxplot(data); ax[0].set_xticks(range(1, len(vs) + 1), [f'{v}\n(n={len(x)})' for v, x in zip(vs, data)]); ax[0].set_ylim(-0.2, 1.02)
        ax[0].set_ylabel('correlation of |a| with the replay implant'); ax[0].set_title('a  Camargo: real shank IMU -> implant', loc='left')
    if len(b):
        grp = list(b.groupby(['dataset', 'task']))
        ax[1].boxplot([g['r_mag'].dropna() for _, g in grp]); ax[1].set_xticks(range(1, len(grp) + 1), [f'{k[0]}\n{k[1]}' for k, _ in grp]); ax[1].set_ylim(-0.2, 1.02)
        ax[1].tick_params(axis='x', labelsize=7); ax[1].set_title('b  Two sensors on one tibia: |a| agreement', loc='left')
    fig.tight_layout(); fig.savefig(os.path.join(out_dir, 'stage1_summary.png'), dpi=130)
    return summ
''')
print('wrote stage01.py')

In [ ]:
#@title Stage 0 — check every dataset (about 2 min)
import sys, importlib
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S
importlib.reload(vimp); importlib.reload(S)
with S.Tee(LOG):
    print(f'\n===== {S.VERSION} | {vimp.VERSION} | stage 0 =====')
    inv = S.inventory(DATA_ROOT, OUT)
    bad = [k for k, v in inv.items() if v['status'] != 'OK']
    print('\nALL DATASETS OK' if not bad else f'\nPROBLEMS: {bad} (paste this output into the chat)')

In [ ]:
#@title Stage 1a — validate on Camargo (1–2 min per subject)
import sys, importlib
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S
importlib.reload(vimp); importlib.reload(S)
with S.Tee(LOG):
    print(f'\n===== stage 1a: Camargo, subjects {SUBJECTS} =====')
    S.stage1a(DATA_ROOT, OUT, SUBJECTS, per_mode=TRIALS_PER_MODE)

In [ ]:
#@title Stage 1b — two sensors on one tibia (about 5–10 min; resumes after a disconnect)
import sys, importlib
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S
importlib.reload(vimp); importlib.reload(S)
import pandas as pd
with S.Tee(LOG):
    print(f'\n===== {S.VERSION} | stage 1b: two sensors on the same tibia (saved after every person; resumes) =====')
    S.stage1b_all(DATA_ROOT, OUT, kneepad_subjects=KNEEPAD_SUBJECTS, weargait_people=WEARGAIT_PEOPLE)

In [ ]:
#@title Summary
import sys, importlib
sys.path.insert(0, '/content/kneil')
import vimp, stage01 as S
importlib.reload(vimp); importlib.reload(S)
from IPython.display import Image, display
with S.Tee(LOG):
    S.summarise(OUT)
display(Image(os.path.join(OUT, 'stage1_summary.png')))
print('\nSend me stage01_log.txt and stage1_summary.png from', OUT)